# Function Vectors via Fully Adaptive All-Head Joint Refinement

This notebook implements the final method. All attention heads are initialized as candidates and ranked by fully adaptive guarded task-level backward refinement. At each refinement step, single-head removals are ranked first; the method proposes an adaptive multi-head deletion, then falls back by halving the batch until a correctness-preserving deletion is found. If no multi-head deletion is safe, it removes the best single head.

The remaining pipeline uses deterministic 60/20/20 train/validation/test splits, 10-shot ICL, successful-only few-shot/zero-shot mean displacement estimation, native-layer steering, and validation-only selection of head count $K$ and intervention strength $\alpha$.


In [1]:
# Run this cell in a fresh kernel. Model-runtime dependencies are pinned for reproducibility.
%pip install -q "transformers==4.53.3" "accelerate==1.8.1" "safetensors==0.5.3" "sentencepiece==0.2.0"
%pip install -q "git+https://github.com/davidbau/baukit.git@9d51abd51ebf29769aecc38c4cbef459b731a36e"

import os
import sys
import gc
import re
import json
import time
import math
import string
import random
import logging
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Any, Optional, Set

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict, get_module
import transformers
transformers.logging.set_verbosity_error()
ALPHA_GENERATION_BATCH_SIZE = 16

FUNCTION_VECTORS_COMMIT = "fb9eac7b6dc707ea1475a717379916007fe448d5"
function_vectors_repo = Path("function_vectors")
if not (function_vectors_repo / ".git").exists():
    !git clone -q https://github.com/ericwtodd/function_vectors.git {function_vectors_repo}
!git -C {function_vectors_repo} checkout -q {FUNCTION_VECTORS_COMMIT}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 79.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.9 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
ALPHA_GENERATION_BATCH_SIZE = 8

## Run configuration

Set the model and seed here, then use **Run All**. Each execution runs the full task suite for **one model** and **one seed**, writing to a seed-specific folder under `outputs/` so runs never overwrite each other.


In [3]:
# ================================================================
# RUN CONFIGURATION
# Override via environment variables (no need to edit this file):
#   FV_SEED=43 ...                     seed: 42, 43, 44
#   FV_MODEL=meta-llama/Llama-3.1-8B   model
#   FV_TASKS=antonym,capitalize        run a subset (smoke test)
# ================================================================

RUN_MODEL = os.environ.get("FV_MODEL", "mistralai/Mistral-7B-v0.3")
RUN_SEED = int(os.environ.get("FV_SEED", "43"))

# Kaggle: pull the HF token from a Kaggle Secret named HF_TOKEN. No-op elsewhere.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("[config] HF token loaded from Kaggle secret")
    except Exception:
        pass

print(f"[config] RUN_MODEL={RUN_MODEL}  RUN_SEED={RUN_SEED}")

# Previously used models: EleutherAI/gpt-j-6b, meta-llama/Llama-3.1-8B, Qwen/Qwen3-4B-Base
# Mistral-7B-v0.3 is gated: `huggingface-cli login` or set HF_TOKEN before launching.


[config] RUN_MODEL=mistralai/Mistral-7B-v0.3  RUN_SEED=43


# 1. Logging and Experiment Tracking


In [4]:
# Configure CUDA Memory Allocation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

class DualLogger:
    """Redirects stdout/stderr to both console and a disk log file safely without recursion."""
    def __init__(self, terminal_stream, filepath: str = "experiment_logs.txt"):
        self.terminal = terminal_stream
        self.filepath = filepath
        self.log_file = open(filepath, "a", encoding="utf-8")

    def write(self, message: str):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def isatty(self) -> bool:
        return getattr(self.terminal, "isatty", lambda: False)()

    def __getattr__(self, attr):
        return getattr(self.terminal, attr)


def setup_logging(log_filename: str = "experiment_logs.txt"):
    """Initializes dual logging by wrapping the current streams (Jupyter/Colab safe)."""
    if not (isinstance(sys.stdout, DualLogger) and sys.stdout.filepath == log_filename):
        sys.stdout = DualLogger(sys.stdout, log_filename)
    if not (isinstance(sys.stderr, DualLogger) and sys.stderr.filepath == log_filename):
        sys.stderr = DualLogger(sys.stderr, log_filename)
    print(f"[*] Logging initialized. Output will be saved to: {log_filename}")


def save_experiment_results(results: Dict[str, Any], filepath: str = "experiment_results.json"):
    """Saves structured experiment results to a JSON file."""
    output_path = Path(filepath)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if not isinstance(data, list):
                data = [data]
        except Exception:
            # Preserve the unreadable file instead of silently overwriting it
            backup = output_path.with_suffix(output_path.suffix + ".bak")
            output_path.replace(backup)
            print(f"[!] Could not parse {filepath}; moved to {backup}")
            data = []
    else:
        data = []

    data.append(results)

    # Write to a temp file first so an interrupted write cannot destroy prior results
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    tmp_path.replace(output_path)

    print(f"[*] Structured results appended to: {filepath}")

# 2. General Utilities and Text Processing


In [5]:

def set_seed(seed: int = 42) -> None:
    """Sets deterministic seeds across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def clean_text(text: Any) -> str:
    """Extracts a case-preserving answer from the first generated line for exact scoring."""
    if not isinstance(text, str):
        return ""
    return text.split("\n", 1)[0].strip()


def normalize_answer(s: str) -> str:
    """Optional relaxed normalization; exact task scoring intentionally does not use this."""
    def remove_articles(text): return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text): return " ".join(text.split())
    def remove_punc(text): return "".join(ch for ch in text if ch not in set(string.punctuation))
    return white_space_fix(remove_articles(remove_punc(s.lower())))


_ANSWER_ID_WARNED = set()

def get_answer_id(query: str, answer: str, tokenizer: AutoTokenizer) -> int:
    """
    Extracts the first token ID of `answer` as it is tokenized in the context of `query`.
    Encoding in context preserves the leading-space merge that BPE tokenizers rely on.
    """
    if isinstance(answer, list):
        answer = answer[0]

    source_ids = tokenizer.encode(query, add_special_tokens=False)
    target_ids = tokenizer.encode(query + answer, add_special_tokens=False)

    # Normal case: the query encoding is a strict prefix of the combined encoding.
    if target_ids[:len(source_ids)] == source_ids:
        new_tokens = target_ids[len(source_ids):]
        if new_tokens:
            return new_tokens[0]

    # BPE re-merged across the boundary: find the longest shared prefix instead.
    k = 0
    while k < len(source_ids) and k < len(target_ids) and source_ids[k] == target_ids[k]:
        k += 1

    key = (getattr(tokenizer, "name_or_path", "?"), answer)
    if key not in _ANSWER_ID_WARNED:
        _ANSWER_ID_WARNED.add(key)
        print(f"[!] get_answer_id: token boundary shift for answer {answer!r} "
              f"(prefix {k}/{len(source_ids)}); using first divergent token.")

    if k < len(target_ids):
        return target_ids[k]

    # Last resort: encode the answer alone.
    standalone = tokenizer.encode(answer, add_special_tokens=False)
    if not standalone:
        raise ValueError(f"Could not derive a target token id for answer {answer!r}")
    return standalone[0]

# 3. Dataset and Prompt Construction

Datasets are split deterministically into 60% training, 20% validation, and 20% test partitions.


In [6]:

class ICLDataset:
    """In-Context Learning Dataset wrapper supporting JSON paths and dictionaries."""
    def __init__(self, dataset: Any):
        if isinstance(dataset, str):
            self.raw_data = pd.read_json(dataset)
        elif isinstance(dataset, dict):
            self.raw_data = pd.DataFrame(dataset)
        elif isinstance(dataset, pd.DataFrame):
            self.raw_data = dataset
        else:
            raise ValueError("Unsupported dataset type.")
        self.raw_data = self.raw_data[["input", "output"]].reset_index(drop=True)

    def __getitem__(self, i: Any) -> Any:
        # np.int64 / np.int32 from np.random.choice must be treated as int
        if isinstance(i, (int, np.integer)):
            return self.raw_data.iloc[int(i)].to_dict()
        elif isinstance(i, (slice, list, np.ndarray, pd.Index)):
            return self.raw_data.iloc[i].to_dict(orient="list")
        elif isinstance(i, str):
            return self.raw_data[i].to_list()
        raise TypeError(f"ICLDataset does not support index of type {type(i)}")

    def __len__(self) -> int:
        return len(self.raw_data)


def split_icl_dataset(dataset: ICLDataset, test_size: float = 0.4, seed: int = 42) -> Dict[str, ICLDataset]:
    """Splits ICL dataset into train, validation, and test sets (60 / 20 / 20)."""
    train, held_out = train_test_split(dataset.raw_data, test_size=test_size, random_state=seed)
    test, valid = train_test_split(held_out, test_size=0.5, random_state=seed)
    return {
        "train": ICLDataset(train.to_dict(orient="list")),
        "valid": ICLDataset(valid.to_dict(orient="list")),
        "test": ICLDataset(test.to_dict(orient="list")),
    }


def load_dataset(task_name: str, root_data_dir: str = "function_vectors/dataset_files",
                 test_size: float = 0.4, seed: int = 42) -> Dict[str, ICLDataset]:
    """Loads benchmark task JSON files across standard categories."""
    data_folders = ["abstractive", "extractive"]
    d_group = list(filter(lambda x: x[1],
                          [(dt, os.path.exists(os.path.join(root_data_dir, dt, f"{task_name}.json")))
                           for dt in data_folders]))
    if not d_group:
        raise FileNotFoundError(f"Task dataset '{task_name}.json' not found under {root_data_dir}.")
    dataset_path = os.path.join(root_data_dir, d_group[0][0], f"{task_name}.json")
    return split_icl_dataset(ICLDataset(dataset_path), test_size=test_size, seed=seed)


def word_pairs_to_prompt_data(
    word_pairs: Dict[str, List[Any]],
    instructions: str = "",
    prefixes: Dict[str, str] = {"input": "Q:", "output": "A:", "instructions": ""},
    separators: Dict[str, str] = {"input": "\n", "output": "\n\n", "instructions": ""},
    query_target_pair: Optional[Dict[str, Any]] = None,
    prepend_bos_token: bool = False,
    bos_string: str = "",
    shuffle_labels: bool = False,
    prepend_space: bool = True
) -> Dict[str, Any]:
    """Converts raw input/output word pairs into standard prompt template dictionaries."""
    prompt_data = {"instructions": instructions, "separators": separators}

    if prepend_bos_token and bos_string:
        prefixes = {k: (v if k != "instructions" else bos_string + v) for k, v in prefixes.items()}
    prompt_data["prefixes"] = prefixes

    if query_target_pair is not None:
        query_target_pair = {k: (v[0] if isinstance(v, list) else v) for k, v in query_target_pair.items()}
    prompt_data["query_target"] = query_target_pair

    inputs = list(word_pairs.get("input", []))
    outputs = list(word_pairs.get("output", []))
    if shuffle_labels and outputs:
        outputs = np.random.permutation(outputs).tolist()

    if prepend_space:
        prompt_data["examples"] = [{"input": " " + str(w1), "output": " " + str(w2)}
                                   for w1, w2 in zip(inputs, outputs)]
        if query_target_pair is not None:
            prompt_data["query_target"] = {k: " " + str(v) for k, v in query_target_pair.items()}
    else:
        prompt_data["examples"] = [{"input": str(w1), "output": str(w2)}
                                   for w1, w2 in zip(inputs, outputs)]

    return prompt_data


def create_prompt(prompt_data: Dict[str, Any], sentence: Optional[str] = None) -> str:
    """Generates the full prompt string from prompt metadata dictionary."""
    if sentence is None and prompt_data["query_target"] is not None:
        sentence = prompt_data["query_target"]["input"]
    if isinstance(sentence, list):
        sentence = sentence[0]

    prompt = (prompt_data["prefixes"]["instructions"]
              + prompt_data["instructions"]
              + prompt_data["separators"]["instructions"])

    for ex in prompt_data["examples"]:
        prompt += prompt_data["prefixes"]["input"] + ex["input"] + prompt_data["separators"]["input"]
        prompt += prompt_data["prefixes"]["output"] + ex["output"] + prompt_data["separators"]["output"]

    return (prompt
            + prompt_data["prefixes"]["input"] + sentence
            + prompt_data["separators"]["input"] + prompt_data["prefixes"]["output"])

# 4. Model Loading and Few-Shot/Zero-Shot Displacement Estimation


In [7]:
def load_gpt_model_and_tokenizer(model_name: str, device: str = None) -> Tuple[Any, Any, Dict[str, Any]]:
    """Loads Causal LM model, tokenizer, and builds architectural hook metadata."""
    print(f"[*] Loading Model: {model_name}")

    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        device_str = "cuda"
    elif torch.backends.mps.is_available():
        dtype = torch.float16
        device_str = "mps"
    else:
        dtype = torch.float32
        device_str = "cpu"

    if device is not None:
        device_str = device

    attn_impl = "eager" if "gpt-j" in model_name.lower() else "sdpa"
    name_lower = model_name.lower()

    if "gpt-j" in name_lower:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        load_kwargs = dict(low_cpu_mem_usage=True, torch_dtype=dtype, attn_implementation=attn_impl)
        if dtype == torch.float16:
            load_kwargs["revision"] = "float16"
        model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs).to(device_str)
        model_config = {
            "n_heads": model.config.n_head,
            "n_layers": model.config.n_layer,
            "resid_dim": model.config.n_embd,
            "name_or_path": model_name,
            "attn_hook_names": [f"transformer.h.{i}.attn.out_proj" for i in range(model.config.n_layer)],
            "layer_hook_names": [f"transformer.h.{i}" for i in range(model.config.n_layer)],
            "prepend_bos": True,
        }
    else:
        trust = "qwen" in name_lower
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        if device_str == "cuda":
            model = AutoModelForCausalLM.from_pretrained(
                model_name, device_map="auto", torch_dtype=dtype,
                trust_remote_code=trust, attn_implementation=attn_impl
            )
            dev_map = getattr(model, "hf_device_map", {})
            offloaded = {str(d) for d in dev_map.values()} & {"cpu", "disk"}
            if offloaded:
                raise RuntimeError(
                    f"Model offloaded to {offloaded} - insufficient GPU memory for {model_name}. "
                    f"Use a larger GPU or a smaller model."
                )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=dtype, trust_remote_code=trust,
                attn_implementation=attn_impl, low_cpu_mem_usage=True
            ).to(device_str)

        model_config = {
            "n_heads": model.config.num_attention_heads,
            "n_layers": model.config.num_hidden_layers,
            "resid_dim": model.config.hidden_size,
            "name_or_path": model_name,
            "attn_hook_names": [f"model.layers.{i}.self_attn.o_proj" for i in range(model.config.num_hidden_layers)],
            "layer_hook_names": [f"model.layers.{i}" for i in range(model.config.num_hidden_layers)],
            "prepend_bos": False,
        }

    model_config["bos_string"] = tokenizer.bos_token if tokenizer.bos_token else ""

    o_proj = get_module(model, model_config["attn_hook_names"][0])
    attn_size = o_proj.in_features if hasattr(o_proj, "in_features") else o_proj.weight.shape[1]
    if attn_size % model_config["n_heads"] != 0:
        raise ValueError(
            f"o_proj in_features {attn_size} not divisible by n_heads {model_config['n_heads']}"
        )
    model_config["head_dim"] = attn_size // model_config["n_heads"]

    dev_map = getattr(model, "hf_device_map", None)
    if dev_map:
        _devs = {str(d) for d in dev_map.values()}
        _bad = _devs & {"cpu", "disk"}
        if _bad:
            raise RuntimeError(
                f"Model offloaded to {_bad} - insufficient GPU memory for {model_name}."
            )
        if len(_devs) > 1:
            print(f"[*] Model sharded across {sorted(_devs)} (multi-GPU device_map).")
    model_config["device"] = next(model.parameters()).device

    model.eval()
    print(
        f"[*] device={model_config['device']}, dtype={dtype}, layers={model_config['n_layers']}, "
        f"heads={model_config['n_heads']}, head_dim={model_config['head_dim']}, "
        f"bos={model_config['bos_string']!r}, prepend_bos={model_config['prepend_bos']}"
    )
    return model, tokenizer, model_config


def _capture_last_token_head_inputs(
    prompt: str,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
) -> torch.Tensor:
    """
    Returns o_proj input at the final prompt token as [n_layers, n_heads, head_dim].
    This is the pre-output-projection attention-head activation used throughout
    both deletion selection and displacement construction.
    """
    device = model_config["device"]
    n_layers = model_config["n_layers"]
    n_heads = model_config["n_heads"]
    head_dim = model_config["head_dim"]

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    result = torch.empty(n_layers, n_heads, head_dim, dtype=torch.float32)

    with TraceDict(model, layers=model_config["attn_hook_names"], retain_input=True) as td:
        with torch.inference_mode():
            model(**inputs)

    for L, layer_name in enumerate(model_config["attn_hook_names"]):
        inp = td[layer_name].input
        if isinstance(inp, tuple):
            inp = inp[0]
        result[L] = inp[0, -1, :].view(n_heads, head_dim).float().cpu()

    del td
    return result


def get_mean_head_displacements(
    dataset: Dict[str, ICLDataset],
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    n_icl_examples: int = 10,
    n_trials: int = 100,
    prefixes: Optional[Dict[str, str]] = None,
    separators: Optional[Dict[str, str]] = None,
    cache_path: Optional[str] = None,
) -> torch.Tensor:
    """
    Estimates the task-level ICL-induced head displacement

        delta_h = a_h^few-shot - a_h^zero-shot

    using matched prompts: the few-shot and zero-shot prompts contain the exact
    same query and differ only by the demonstrations. Averaging is performed
    only here, after head-selection logic; selection itself remains prompt-specific.
    """
    if cache_path is not None and os.path.exists(cache_path):
        print(f"[*] Loading cached mean head displacements from {cache_path}")
        return torch.load(cache_path, map_location="cpu")

    n_layers = model_config["n_layers"]
    n_heads = model_config["n_heads"]
    head_dim = model_config["head_dim"]
    n_train = len(dataset["train"])

    sample_size = min(n_icl_examples + 1, n_train)
    actual_shots = sample_size - 1
    if actual_shots < 1:
        raise ValueError(f"Training split too small ({n_train}) for {n_icl_examples}-shot prompts.")

    pd_kwargs = {
        "prepend_bos_token": model_config["prepend_bos"],
        "bos_string": model_config["bos_string"],
    }
    if prefixes is not None:
        pd_kwargs["prefixes"] = prefixes
    if separators is not None:
        pd_kwargs["separators"] = separators

    displacement_sum = torch.zeros(n_layers, n_heads, head_dim, dtype=torch.float64)

    for _ in tqdm(
        range(n_trials),
        desc="Calculating Mean ICL Head Displacements (Train Only)",
        mininterval=5.0,
    ):
        sampled_indices = np.random.choice(n_train, sample_size, replace=False)
        demo_indices = sampled_indices[:actual_shots]
        query_index = sampled_indices[actual_shots]

        demos = dataset["train"][demo_indices]
        query_pair = dataset["train"][query_index]

        prompt_fewshot = create_prompt(
            word_pairs_to_prompt_data(demos, query_target_pair=query_pair, **pd_kwargs)
        )
        prompt_zeroshot = create_prompt(
            word_pairs_to_prompt_data(
                {"input": [], "output": []},
                query_target_pair=query_pair,
                **pd_kwargs,
            )
        )

        fewshot_acts = _capture_last_token_head_inputs(
            prompt_fewshot, model, model_config, tokenizer
        )
        zeroshot_acts = _capture_last_token_head_inputs(
            prompt_zeroshot, model, model_config, tokenizer
        )

        displacement_sum += (fewshot_acts - zeroshot_acts).double()

        del fewshot_acts, zeroshot_acts

    mean_displacements = (displacement_sum / float(n_trials)).float()

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        torch.save(mean_displacements, cache_path)
        print(f"[*] Cached mean head displacements to {cache_path}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return mean_displacements


def get_successful_mean_head_displacements(
    dataset: Dict[str, ICLDataset],
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    n_icl_examples: int = 10,
    n_trials: int = 100,
    prefixes: Optional[Dict[str, str]] = None,
    separators: Optional[Dict[str, str]] = None,
    seed: int = 42,
    cache_path: Optional[str] = None,
) -> torch.Tensor:
    """
    Estimates E[a_fewshot - a_zeroshot] using only TRAIN prompts on which
    clean few-shot greedy generation exactly matches the target.
    """
    if cache_path is not None and os.path.exists(cache_path):
        payload = torch.load(cache_path, map_location="cpu")
        if isinstance(payload, dict) and payload.get("version") == 1:
            print(f"[*] Loading successful-only mean displacements from {cache_path}")
            return payload["mean_displacements"].float().cpu()

    n_layers = model_config["n_layers"]
    n_heads = model_config["n_heads"]
    head_dim = model_config["head_dim"]
    n_train = len(dataset["train"])

    pd_kwargs = {
        "prepend_bos_token": model_config["prepend_bos"],
        "bos_string": model_config["bos_string"],
    }
    if prefixes is not None:
        pd_kwargs["prefixes"] = prefixes
    if separators is not None:
        pd_kwargs["separators"] = separators

    rng = np.random.default_rng(seed + 100003)
    displacement_sum = torch.zeros(
        n_layers, n_heads, head_dim, dtype=torch.float64
    )
    accepted = 0
    attempts = 0
    target_order = rng.permutation(n_train)
    cursor = 0
    max_attempts = max(n_train * 10, n_trials * 60)

    pbar = tqdm(
        total=n_trials,
        desc="Successful-only Mean ICL Head Displacements (Train Only)",
        mininterval=5.0,
    )

    while accepted < n_trials and attempts < max_attempts:
        if cursor >= len(target_order):
            target_order = rng.permutation(n_train)
            cursor = 0

        target_idx = int(target_order[cursor])
        cursor += 1
        attempts += 1

        remaining = np.array(
            [i for i in range(n_train) if i != target_idx], dtype=int
        )
        demo_indices = rng.choice(
            remaining, size=n_icl_examples, replace=False
        )
        demos = dataset["train"][demo_indices]
        query_pair = dataset["train"][target_idx]
        target = query_pair["output"]
        target_str = str(target[0] if isinstance(target, list) else target)

        prompt_fewshot = create_prompt(
            word_pairs_to_prompt_data(
                demos, query_target_pair=query_pair, **pd_kwargs
            )
        )

        inputs = tokenizer(
            [prompt_fewshot], return_tensors="pt"
        ).to(model_config["device"])
        prompt_len = inputs.input_ids.shape[1]
        max_new_tokens = max(
            3,
            len(
                tokenizer.encode(
                    " " + target_str.strip(),
                    add_special_tokens=False,
                )
            ),
        )
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = clean_text(
            tokenizer.decode(
                generated_ids[0][prompt_len:],
                skip_special_tokens=True,
            )
        )
        if generated != clean_text(target_str):
            continue

        prompt_zeroshot = create_prompt(
            word_pairs_to_prompt_data(
                {"input": [], "output": []},
                query_target_pair=query_pair,
                **pd_kwargs,
            )
        )

        fewshot_acts = _capture_last_token_head_inputs(
            prompt_fewshot, model, model_config, tokenizer
        )
        zeroshot_acts = _capture_last_token_head_inputs(
            prompt_zeroshot, model, model_config, tokenizer
        )
        displacement_sum += (fewshot_acts - zeroshot_acts).double()

        accepted += 1
        pbar.update(1)
        pbar.set_postfix(attempts=attempts)

        del fewshot_acts, zeroshot_acts

    pbar.close()

    if accepted < n_trials:
        raise RuntimeError(
            f"Could collect only {accepted} / {n_trials} successful "
            f"mean-displacement prompts after {attempts} attempts."
        )

    mean_displacements = (displacement_sum / float(accepted)).float()

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        torch.save(
            {
                "version": 1,
                "seed": int(seed),
                "n_shots": int(n_icl_examples),
                "n_trials": int(n_trials),
                "attempts": int(attempts),
                "success_only": True,
                "mean_displacements": mean_displacements.cpu(),
            },
            cache_path,
        )
        print(
            f"[*] Cached successful-only mean displacements to {cache_path}"
        )

    return mean_displacements



# 5. Head Displacement Projection and Layer-Wise Steering


In [8]:
def project_head_displacement_vectors(
    core_heads: List[Tuple[int, int]],
    head_displacements: torch.Tensor,
    model: Any,
    model_config: Dict[str, Any],
    device: torch.device,
    dtype: torch.dtype,
) -> Dict[Tuple[int, int], torch.Tensor]:
    """
    Projects each selected head displacement with the linear part of o_proj only:

        v_{L,h} = W_{O,L,h} @ (a_fewshot - a_zeroshot)

    The o_proj bias is intentionally excluded. A displacement is a difference,
    so any affine bias cancels and must not be injected.
    """
    head_dim = head_displacements.size(-1)
    projected: Dict[Tuple[int, int], torch.Tensor] = {}

    for L, H in core_heads:
        proj_module = get_module(model, model_config["attn_hook_names"][L])
        weight = proj_module.weight
        start = H * head_dim
        end = start + head_dim

        if end > weight.shape[1]:
            raise ValueError(
                f"Head slice [{start}:{end}] exceeds o_proj input width {weight.shape[1]} "
                f"for layer {L}, head {H}."
            )

        delta = head_displacements[L, H].to(device=weight.device, dtype=weight.dtype)
        weight_slice = weight[:, start:end]

        with torch.inference_mode():
            vec = F.linear(delta.reshape(1, -1), weight_slice, bias=None)

        projected[(L, H)] = vec.reshape(1, 1, -1).to(
            device=device, dtype=dtype
        ).detach()

    return projected


def compute_layerwise_function_vectors(
    core_heads: List[Tuple[int, int]],
    mean_displacements: torch.Tensor,
    model: Any,
    model_config: Dict[str, Any],
    device: torch.device,
    dtype: torch.dtype,
) -> Dict[int, torch.Tensor]:
    """
    Builds the final layer-wise steering vectors from mean ICL-induced
    few-shot/zero-shot displacements of the selected heads.
    """
    if not core_heads:
        return {}

    head_vectors = project_head_displacement_vectors(
        core_heads=core_heads,
        head_displacements=mean_displacements,
        model=model,
        model_config=model_config,
        device=device,
        dtype=dtype,
    )

    layer_fvs: Dict[int, torch.Tensor] = {}
    for (L, H), vec in head_vectors.items():
        if L not in layer_fvs:
            layer_fvs[L] = torch.zeros_like(vec)
        layer_fvs[L] = layer_fvs[L] + vec

    return {L: v.detach() for L, v in layer_fvs.items()}


def layerwise_fv_intervention(
    sentence: List[str],
    core_heads: List[Tuple[int, int]],
    mean_displacements: torch.Tensor,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    generate_str: bool = True,
    alpha: float = 1.0,
    layer_fvs: Optional[Dict[int, torch.Tensor]] = None,
    compute_clean: bool = True,
    max_new_tokens: int = 16,
) -> Tuple[str, str]:
    """
    Zero-shot generation with additive residual-stream injection:

        r_L <- r_L + alpha * sum_h W_{O,L,h} E[a_fewshot-a_zeroshot].

    The intervention is applied at each selected head's native layer.
    """
    device = model_config["device"]
    dtype = next(model.parameters()).dtype
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]

    with torch.inference_mode():
        if generate_str and compute_clean:
            base_out = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )
            clean_output = tokenizer.decode(
                base_out[0][prompt_len:], skip_special_tokens=True
            )
        else:
            clean_output = ""

    if not core_heads:
        return clean_output, clean_output

    if layer_fvs is None:
        layer_fvs = compute_layerwise_function_vectors(
            core_heads=core_heads,
            mean_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            device=device,
            dtype=dtype,
        )

    hook_layer_names = [
        model_config["layer_hook_names"][L] for L in layer_fvs.keys()
    ]
    layer_name_to_idx = {
        name: L for L, name in enumerate(model_config["layer_hook_names"])
    }

    def add_layerwise_fv_hook(output, layer_name, inputs):
        is_tuple = isinstance(output, tuple)
        hidden_states = output[0] if is_tuple else output

        layer_idx = layer_name_to_idx.get(layer_name)
        if layer_idx in layer_fvs:
            fv = layer_fvs[layer_idx].reshape(1, -1).to(
                device=hidden_states.device, dtype=hidden_states.dtype
            )
            hidden_states = hidden_states.clone()
            hidden_states[:, -1, :] = (
                hidden_states[:, -1, :] + float(alpha) * fv
            )

            if is_tuple:
                return (hidden_states,) + output[1:]
            return hidden_states

        return output

    with TraceDict(
        model,
        layers=hook_layer_names,
        edit_output=add_layerwise_fv_hook,
    ):
        with torch.inference_mode():
            if generate_str:
                interv_out = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    pad_token_id=tokenizer.eos_token_id,
                )
                intervention_output = tokenizer.decode(
                    interv_out[0][prompt_len:],
                    skip_special_tokens=True,
                )
            else:
                intervention_output = ""

    return clean_output, intervention_output


def layerwise_fv_intervention_batch(
    sentence: List[str],
    alphas: List[float],
    core_heads: List[Tuple[int, int]],
    mean_displacements: torch.Tensor,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    layer_fvs: Optional[Dict[int, torch.Tensor]] = None,
    max_new_tokens: int = 16,
) -> List[str]:
    """Generates one shared zero-shot prompt for several alpha values."""
    alpha_values = [float(a) for a in alphas]
    if not alpha_values:
        return []

    device = model_config["device"]
    dtype = next(model.parameters()).dtype
    batched_sentences = sentence * len(alpha_values)
    inputs = tokenizer(
        batched_sentences, return_tensors="pt", padding=True
    ).to(device)
    prompt_len = inputs.input_ids.shape[1]

    if layer_fvs is None and core_heads:
        layer_fvs = compute_layerwise_function_vectors(
            core_heads=core_heads,
            mean_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            device=device,
            dtype=dtype,
        )
    layer_fvs = layer_fvs or {}

    hook_layer_names = [
        model_config["layer_hook_names"][L] for L in layer_fvs.keys()
    ]
    layer_name_to_idx = {
        name: L for L, name in enumerate(model_config["layer_hook_names"])
    }
    alpha_tensor = torch.tensor(
        alpha_values, device=device, dtype=dtype
    ).reshape(-1, 1)

    def add_layerwise_fv_hook(output, layer_name, inputs):
        is_tuple = isinstance(output, tuple)
        hidden_states = output[0] if is_tuple else output

        layer_idx = layer_name_to_idx.get(layer_name)
        if layer_idx in layer_fvs:
            fv = layer_fvs[layer_idx].reshape(1, -1).to(
                device=hidden_states.device, dtype=hidden_states.dtype
            )
            scales = alpha_tensor.to(
                device=hidden_states.device, dtype=hidden_states.dtype
            )

            hidden_states = hidden_states.clone()
            hidden_states[:, -1, :] = (
                hidden_states[:, -1, :] + scales * fv
            )

            if is_tuple:
                return (hidden_states,) + output[1:]
            return hidden_states

        return output

    if hook_layer_names:
        with TraceDict(
            model,
            layers=hook_layer_names,
            edit_output=add_layerwise_fv_hook,
        ):
            with torch.inference_mode():
                intervention_outputs = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    pad_token_id=tokenizer.eos_token_id,
                )
    else:
        with torch.inference_mode():
            intervention_outputs = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )

    return tokenizer.batch_decode(
        intervention_outputs[:, prompt_len:],
        skip_special_tokens=True,
    )


def validate_head_fv_accuracy(
    core_heads: List[Tuple[int, int]],
    mean_displacements: torch.Tensor,
    dataset: Dict[str, ICLDataset],
    split_name: str,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    prefixes: Dict[str, str],
    separators: Dict[str, str],
    filter_set: Optional[np.ndarray] = None,
    alpha: float = 1.0,
    layer_fvs: Optional[Dict[int, torch.Tensor]] = None,
) -> float:
    """Evaluates zero-shot accuracy with displacement-based FV injection."""
    target_data = dataset[split_name]
    if filter_set is None:
        filter_set = np.arange(min(10, len(target_data)))

    if layer_fvs is None and core_heads:
        layer_fvs = compute_layerwise_function_vectors(
            core_heads=core_heads,
            mean_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            device=model_config["device"],
            dtype=next(model.parameters()).dtype,
        )

    scores = []
    for j in filter_set:
        word_pairs_eval = target_data[int(j)]
        query, target = word_pairs_eval["input"], word_pairs_eval["output"]
        if isinstance(target, list):
            target = target[0]

        prompt_data = word_pairs_to_prompt_data(
            {"input": [], "output": []},
            query_target_pair={"input": query, "output": ""},
            prepend_bos_token=model_config["prepend_bos"],
            bos_string=model_config["bos_string"],
            prefixes=prefixes,
            separators=separators,
        )
        sentence = [create_prompt(prompt_data)]
        eval_max_new_tokens = max(
            3,
            len(
                tokenizer.encode(
                    " " + str(target).strip(),
                    add_special_tokens=False,
                )
            ),
        )

        _, interv_out = layerwise_fv_intervention(
            sentence=sentence,
            core_heads=core_heads,
            mean_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            tokenizer=tokenizer,
            generate_str=True,
            alpha=alpha,
            layer_fvs=layer_fvs,
            compute_clean=False,
            max_new_tokens=eval_max_new_tokens,
        )

        scores.append(
            1.0 if clean_text(interv_out) == clean_text(target) else 0.0
        )

    return float(np.mean(scores)) if scores else 0.0


def validate_head_fv_accuracies(
    alphas: List[float],
    core_heads: List[Tuple[int, int]],
    mean_displacements: torch.Tensor,
    dataset: Dict[str, ICLDataset],
    split_name: str,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    prefixes: Dict[str, str],
    separators: Dict[str, str],
    filter_set: Optional[np.ndarray] = None,
    layer_fvs: Optional[Dict[int, torch.Tensor]] = None,
) -> Dict[float, float]:
    """Evaluates the same displacement-based FV for several alpha values."""
    alpha_values = [float(a) for a in alphas]
    if not alpha_values:
        return {}

    target_data = dataset[split_name]
    if filter_set is None:
        filter_set = np.arange(min(10, len(target_data)))

    if layer_fvs is None and core_heads:
        layer_fvs = compute_layerwise_function_vectors(
            core_heads=core_heads,
            mean_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            device=model_config["device"],
            dtype=next(model.parameters()).dtype,
        )

    correct = {a: 0.0 for a in alpha_values}
    num_examples = 0

    for j in filter_set:
        word_pairs_eval = target_data[int(j)]
        query, target = word_pairs_eval["input"], word_pairs_eval["output"]
        if isinstance(target, list):
            target = target[0]

        prompt_data = word_pairs_to_prompt_data(
            {"input": [], "output": []},
            query_target_pair={"input": query, "output": ""},
            prepend_bos_token=model_config["prepend_bos"],
            bos_string=model_config["bos_string"],
            prefixes=prefixes,
            separators=separators,
        )
        sentence = [create_prompt(prompt_data)]
        eval_max_new_tokens = max(
            3,
            len(
                tokenizer.encode(
                    " " + str(target).strip(),
                    add_special_tokens=False,
                )
            ),
        )
        expected_clean = clean_text(target)

        for start in range(
            0, len(alpha_values), ALPHA_GENERATION_BATCH_SIZE
        ):
            alpha_batch = alpha_values[
                start:start + ALPHA_GENERATION_BATCH_SIZE
            ]
            intervention_outputs = layerwise_fv_intervention_batch(
                sentence=sentence,
                alphas=alpha_batch,
                core_heads=core_heads,
                mean_displacements=mean_displacements,
                model=model,
                model_config=model_config,
                tokenizer=tokenizer,
                layer_fvs=layer_fvs,
                max_new_tokens=eval_max_new_tokens,
            )

            for alpha, intervention_output in zip(
                alpha_batch, intervention_outputs
            ):
                if clean_text(intervention_output) == expected_clean:
                    correct[alpha] += 1.0

        num_examples += 1

    if num_examples == 0:
        return {a: 0.0 for a in alpha_values}

    return {
        a: correct[a] / num_examples
        for a in alpha_values
    }


def get_few_shot_filter_set(
    dataset: Dict[str, ICLDataset],
    split_name: str,
    n_shots: int,
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    prefixes: Dict[str, str],
    separators: Dict[str, str],
    seed: int = 42,
) -> np.ndarray:
    """Finds examples on which the base model succeeds with n-shot ICL."""
    print(
        f"[*] Calculating {n_shots}-Shot Baseline Accuracy "
        f"on '{split_name.upper()}' split..."
    )
    target_data = dataset[split_name]
    device = model_config["device"]
    successful_indices = []
    rng = np.random.default_rng(seed)

    for j in tqdm(
        range(len(target_data)),
        desc=f"Finding Accurate ICL Prompts ({split_name})",
    ):
        word_pairs_target = target_data[j]
        query, target = (
            word_pairs_target["input"],
            word_pairs_target["output"],
        )
        target_str = target[0] if isinstance(target, list) else target
        target_clean = clean_text(target_str)
        eval_max_new_tokens = max(
            3,
            len(
                tokenizer.encode(
                    " " + str(target_str).strip(),
                    add_special_tokens=False,
                )
            ),
        )

        word_pairs_train = (
            dataset["train"][
                rng.choice(
                    len(dataset["train"]),
                    n_shots,
                    replace=False,
                )
            ]
            if n_shots > 0
            else {"input": [], "output": []}
        )

        prompt_data = word_pairs_to_prompt_data(
            word_pairs_train,
            query_target_pair={"input": query, "output": ""},
            prepend_bos_token=model_config["prepend_bos"],
            bos_string=model_config["bos_string"],
            prefixes=prefixes,
            separators=separators,
        )
        sentence = [create_prompt(prompt_data)]
        inputs = tokenizer(sentence, return_tensors="pt").to(device)
        prompt_len = inputs.input_ids.shape[1]

        with torch.inference_mode():
            base_out = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=eval_max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )
            clean_output = clean_text(
                tokenizer.decode(
                    base_out[0][prompt_len:],
                    skip_special_tokens=True,
                )
            )

        if clean_output == target_clean:
            successful_indices.append(j)

    print(
        f"[*] Found {len(successful_indices)} / {len(target_data)} "
        f"successful baseline prompts on '{split_name}'."
    )
    return np.array(successful_indices, dtype=int)


# 6. Prompt-Level Discovery Utilities

This helper implementation is retained for compatibility with earlier experiments but is not used by the final method. The final method initializes joint refinement directly from all model heads.


In [9]:
def iterative_head_knockout_dynamic_fast_vectorized(
    prompt_icl: str,
    prompt_neutral: str,
    target_str: str,
    model: Any,
    tokenizer: Any,
    model_config: Dict[str, Any],
    initial_start_layer: int = 8,
    initial_end_layer: int = 10,
    eval_batch_size: int = 32,
) -> Dict[str, Any]:
    """
    Legacy prompt-level causal head-discovery utility retained for
    compatibility with earlier experiments.

    A selected head is replaced at the query position:
        a_fewshot -> a_zeroshot
    removing the ICL-induced displacement
        delta = a_fewshot - a_zeroshot.

    This utility is not called by the final all-head joint-refinement method.
    """
    device = model_config["device"]
    dtype = next(model.parameters()).dtype
    n_layers = model_config["n_layers"]
    n_heads = model_config["n_heads"]
    head_dim = model_config["head_dim"]

    def target_ids_for_prompt(prompt: str) -> List[int]:
        target_text = " " + target_str.strip()
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
        completion_ids = tokenizer.encode(
            prompt + target_text, add_special_tokens=False
        )
        if completion_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(
                "Target text changes tokenization across the prompt boundary."
            )
        ids = completion_ids[len(prompt_ids):]
        if not ids:
            raise ValueError(f"Empty target tokenization for {target_str!r}.")
        return ids

    target_ids_icl = target_ids_for_prompt(prompt_icl)
    target_ids_zs = target_ids_for_prompt(prompt_neutral)

    inputs_icl = tokenizer(prompt_icl, return_tensors="pt").to(device)
    inputs_zs = tokenizer(prompt_neutral, return_tensors="pt").to(device)
    prompt_len_icl = inputs_icl.input_ids.shape[1]
    prompt_len_zs = inputs_zs.input_ids.shape[1]

    # ---------------------------------------------------------------
    # Capture clean prompt-specific head activations BEFORE any edit.
    # These matched activations define delta = few-shot - zero-shot.
    # ---------------------------------------------------------------
    fewshot_cache: Dict[int, torch.Tensor] = {}
    zeroshot_cache: Dict[int, torch.Tensor] = {}
    capture_handles = []

    def capture_o_proj_input(layer_idx: int, cache: Dict[int, torch.Tensor]):
        def hook(module, args):
            inp = args[0]
            cache[layer_idx] = inp[0, -1, :].detach().clone()
        return hook

    for L in range(n_layers):
        module = get_module(model, model_config["attn_hook_names"][L])
        capture_handles.append(
            module.register_forward_pre_hook(
                capture_o_proj_input(L, fewshot_cache)
            )
        )
    with torch.inference_mode():
        model(**inputs_icl)
    for handle in capture_handles:
        handle.remove()
    capture_handles.clear()

    for L in range(n_layers):
        module = get_module(model, model_config["attn_hook_names"][L])
        capture_handles.append(
            module.register_forward_pre_hook(
                capture_o_proj_input(L, zeroshot_cache)
            )
        )
    with torch.inference_mode():
        model(**inputs_zs)
    for handle in capture_handles:
        handle.remove()
    capture_handles.clear()

    prompt_displacements = torch.empty(
        n_layers, n_heads, head_dim, dtype=torch.float32
    )
    for L in range(n_layers):
        f = fewshot_cache[L].view(n_heads, head_dim).float().cpu()
        z = zeroshot_cache[L].view(n_heads, head_dim).float().cpu()
        prompt_displacements[L] = f - z

    # ---------------------------------------------------------------
    # Teacher-forced full ICL input for Phase 1 scoring.
    # ---------------------------------------------------------------
    icl_target_tensor = torch.tensor(
        [target_ids_icl], device=device, dtype=inputs_icl.input_ids.dtype
    )
    full_icl_ids = torch.cat(
        [inputs_icl.input_ids, icl_target_tensor], dim=1
    )
    full_icl_mask = torch.cat(
        [
            inputs_icl.attention_mask,
            torch.ones_like(icl_target_tensor),
        ],
        dim=1,
    )
    icl_start_idx = prompt_len_icl - 1
    icl_end_idx = icl_start_idx + len(target_ids_icl)

    with torch.inference_mode():
        clean_icl_logits = model(
            input_ids=full_icl_ids,
            attention_mask=full_icl_mask,
        ).logits

    clean_target_logits = clean_icl_logits[
        0, icl_start_idx:icl_end_idx, :
    ]
    clean_top1 = clean_target_logits.argmax(dim=-1).tolist()
    if clean_top1 != target_ids_icl:
        print("⚠️ Model failed on Clean Base! Skipping.")
        return {
            "status": "clean_base_fail",
            "heads": None,
            "head_drops": None,
        }

    # ===============================================================
    # PHASE 1: Natural ICL deletion
    # ===============================================================
    phase1_state = {
        "persistent_patched": [set() for _ in range(n_layers)],
        "batch_candidates": None,
    }

    def phase1_hook(layer_idx: int):
        def hook(module, args):
            inp = args[0]
            # During cached autoregressive decoding seq_len is normally 1.
            # Phase 1 deliberately edits the query-position computation only.
            if inp.shape[1] < prompt_len_icl:
                return args

            original_shape = inp.shape
            bsz, seq_len, _ = original_shape
            edited = inp.clone().view(
                bsz, seq_len, n_heads, head_dim
            )

            neutral = zeroshot_cache[layer_idx].view(
                1, n_heads, head_dim
            )

            persistent = phase1_state["persistent_patched"][layer_idx]
            if persistent:
                heads = list(persistent)
                edited[:, prompt_len_icl - 1, heads, :] = (
                    neutral[:, heads, :]
                )

            batch_candidates = phase1_state["batch_candidates"]
            if batch_candidates is not None:
                for row_idx, (cand_l, cand_h) in enumerate(
                    batch_candidates
                ):
                    if row_idx >= bsz:
                        break
                    if cand_l == layer_idx:
                        edited[
                            row_idx,
                            prompt_len_icl - 1,
                            cand_h,
                            :,
                        ] = neutral[0, cand_h, :]

            return (edited.view(original_shape),)

        return hook

    phase1_handles = [
        get_module(
            model, model_config["attn_hook_names"][L]
        ).register_forward_pre_hook(phase1_hook(L))
        for L in range(n_layers)
    ]

    def phase1_eval_single() -> Tuple[bool, float]:
        phase1_state["batch_candidates"] = None
        with torch.inference_mode():
            logits = model(
                input_ids=full_icl_ids,
                attention_mask=full_icl_mask,
            ).logits[0, icl_start_idx:icl_end_idx, :]

        top1 = logits.argmax(dim=-1).tolist()
        log_probs = F.log_softmax(logits, dim=-1)
        token_scores = [
            log_probs[t, tid]
            for t, tid in enumerate(target_ids_icl)
        ]
        mean_log_prob = torch.stack(token_scores).mean().item()
        return top1 == target_ids_icl, mean_log_prob

    def phase1_eval_candidates(
        candidates: List[Tuple[int, int]],
    ) -> List[Tuple[Tuple[int, int], bool, float]]:
        results = []
        target_t = torch.tensor(
            target_ids_icl, device=device
        )

        for start in range(0, len(candidates), eval_batch_size):
            chunk = candidates[start:start + eval_batch_size]
            if not chunk:
                continue

            cur_bsz = len(chunk)
            phase1_state["batch_candidates"] = chunk
            batch_ids = full_icl_ids.repeat(cur_bsz, 1)
            batch_mask = full_icl_mask.repeat(cur_bsz, 1)

            with torch.inference_mode():
                logits = model(
                    input_ids=batch_ids,
                    attention_mask=batch_mask,
                ).logits[:, icl_start_idx:icl_end_idx, :]

            top1 = logits.argmax(dim=-1)
            correct = (top1 == target_t).all(dim=-1)
            log_probs = F.log_softmax(logits, dim=-1)

            for row, head in enumerate(chunk):
                token_scores = torch.stack(
                    [
                        log_probs[row, t, tid]
                        for t, tid in enumerate(target_ids_icl)
                    ]
                )
                results.append(
                    (
                        head,
                        bool(correct[row].item()),
                        float(token_scores.mean().item()),
                    )
                )

        phase1_state["batch_candidates"] = None
        return results

    def phase1_generate() -> str:
        phase1_state["batch_candidates"] = None
        with torch.inference_mode():
            out = model.generate(
                **inputs_icl,
                do_sample=False,
                max_new_tokens=max(len(target_ids_icl), 3),
                pad_token_id=tokenizer.eos_token_id,
            )
        return clean_text(
            tokenizer.decode(
                out[0][prompt_len_icl:],
                skip_special_tokens=True,
            )
        )

    try:
        search_start = max(0, int(initial_start_layer))
        search_end = min(n_layers, int(initial_end_layer))
        target_clean = clean_text(target_str)

        # Expand the search region until BOTH teacher-forced correctness
        # and actual autoregressive generation survive neutralizing heads
        # outside the region.
        while True:
            for L in range(n_layers):
                phase1_state["persistent_patched"][L] = (
                    set(range(n_heads))
                    if (L < search_start or L >= search_end)
                    else set()
                )

            tf_ok, _ = phase1_eval_single()
            gen_ok = tf_ok and (phase1_generate() == target_clean)
            if gen_ok:
                break

            if search_start == 0 and search_end == n_layers:
                break

            search_start = max(0, search_start - 1)
            search_end = min(n_layers, search_end + 1)

        active_pool = {
            (L, H)
            for L in range(search_start, search_end)
            for H in range(n_heads)
            if H not in phase1_state["persistent_patched"][L]
        }

        phase1_history = [
            (
                active_pool.copy(),
                [s.copy() for s in phase1_state["persistent_patched"]],
            )
        ]

        print(
            "      🚀 [Phase 1] Natural ICL displacement deletion "
            "(a_fewshot -> a_zeroshot)..."
        )

        while active_pool:
            stats = phase1_eval_candidates(list(active_pool))
            removable = [
                (head, score)
                for head, is_correct, score in stats
                if is_correct
            ]
            if not removable:
                break

            removable.sort(key=lambda x: x[1], reverse=True)
            batch_size = (
                max(1, int(len(active_pool) * 0.3))
                if len(active_pool) >= 4
                else 1
            )
            batch_size = min(batch_size, len(removable))
            applied = False

            while batch_size >= 1:
                batch_to_remove = [
                    head for head, _ in removable[:batch_size]
                ]

                for L, H in batch_to_remove:
                    phase1_state["persistent_patched"][L].add(H)

                tf_ok, _ = phase1_eval_single()
                if tf_ok:
                    for head in batch_to_remove:
                        active_pool.remove(head)
                    phase1_history.append(
                        (
                            active_pool.copy(),
                            [
                                s.copy()
                                for s in phase1_state[
                                    "persistent_patched"
                                ]
                            ],
                        )
                    )
                    applied = True
                    break

                for L, H in batch_to_remove:
                    phase1_state["persistent_patched"][L].remove(H)

                batch_size //= 2

            if not applied:
                break

        # Teacher forcing can hide an autoregressive failure. Roll back
        # committed Phase-1 batches until generation is correct.
        phase1_output = phase1_generate()
        if phase1_output != target_clean:
            for hist_idx in range(
                len(phase1_history) - 2, -1, -1
            ):
                pool_snapshot, patched_snapshot = phase1_history[hist_idx]
                active_pool = pool_snapshot.copy()
                phase1_state["persistent_patched"] = [
                    s.copy() for s in patched_snapshot
                ]
                phase1_output = phase1_generate()
                if phase1_output == target_clean:
                    restored = (
                        len(phase1_history) - 1 - hist_idx
                    )
                    print(
                        f"      ↩️ [Phase 1 Rollback] "
                        f"Restored {restored} removal batch(es)."
                    )
                    break

        if phase1_output != target_clean:
            print(
                "      ❌ [Phase 1 Rejected] Could not preserve "
                "autoregressive target generation."
            )
            return {
                "status": "generation_fail",
                "heads": None,
                "head_drops": None,
            }

        phase1_heads = sorted(active_pool)
        print(
            f"      ✔️ [Phase 1 Finished] Remaining Heads: "
            f"{len(phase1_heads)}"
        )

        # -----------------------------------------------------------
        # Score the final prompt-level survivors in the natural ICL
        # intervention space. This legacy utility is not used by the
        # final all-head joint-refinement method.
        # -----------------------------------------------------------
        phase1_ok, phase1_base_score = phase1_eval_single()
        if not phase1_ok:
            print(
                "      ❌ [Phase 1 Rejected] Final teacher-forced "
                "target is no longer correct."
            )
            return {
                "status": "generation_fail",
                "heads": None,
                "head_drops": None,
            }

        head_drops: Dict[Tuple[int, int], float] = {}
        if phase1_heads:
            final_stats = phase1_eval_candidates(
                phase1_heads
            )
            for head, _, ablated_score in final_stats:
                head_drops[head] = float(
                    phase1_base_score
                    - ablated_score
                )

        important_heads = sorted(
            phase1_heads,
            key=lambda h: head_drops.get(h, 0.0),
            reverse=True,
        )

        print(
            "      ➡️ [Legacy Candidate Set] "
            "Using all prompt-level survivors as candidates."
        )
        print(
            f"      🎯 Expected Target   : "
            f"'{target_str.strip()}'"
        )
        print(
            f"      🤖 Model Final Output: "
            f"'{phase1_output}'"
        )
        print(
            f"      ✅ [Done] Phase-1 Heads: "
            f"{len(important_heads)} | "
            f"Score: signed Δ mean log-p (nats/token)"
        )

        return {
            "status": "success",
            "heads": important_heads,
            "head_drops": head_drops,
            "phase1_heads": important_heads,
            "phase2_heads": None,
            "phase2_enabled": False,
            "head_score_name": "signed_mean_logprob_drop",
            "head_score_unit": "nats_per_target_token",
            "selection_signal": "phase1_a_fewshot_to_a_zeroshot",
        }

    finally:
        for handle in phase1_handles:
            handle.remove()

        del fewshot_cache, zeroshot_cache, prompt_displacements

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# 7. Reporting and Visualization


In [10]:
def print_heads_beautifully(
    heads_list: List[Tuple[int, int]],
    title: str = "EXTRACTED HEADS",
    head_scores: Optional[Dict[Tuple[int, int], float]] = None,
    score_label: str = "Score"
):
    """Formatted terminal output for extracted attention heads with impact scores."""
    print(f"\n   💎 {title} ({len(heads_list)} Heads):")
    if not heads_list:
        print("      [ None ]")
        return

    if head_scores is not None:
        sorted_heads = sorted(heads_list, key=lambda h: head_scores.get(h, 0.0), reverse=True)
        for rank, (L, H) in enumerate(sorted_heads, 1):
            score = head_scores.get((L, H), 0.0)
            print(f"      {rank:02d}. Layer {L:02d}, Head {H:02d}  ->  {score_label}: {score:6.2f}%")
    else:
        layer_dict = defaultdict(list)
        for L, H in heads_list:
            layer_dict[L].append(H)
        for L in sorted(layer_dict.keys()):
            heads_str = ", ".join([f"H{H:02d}" for H in sorted(layer_dict[L])])
            print(f"      • Layer {L:02d} : [ {heads_str} ]")

# 8. Fully Adaptive All-Head Joint Refinement, Validation Tuning, and Test Evaluation

Successful TRAIN prompts are used to rank all model heads jointly with the same adaptive guarded deletion schedule throughout the ranking. Validation selects $K$ and $\alpha$; the frozen configuration is then evaluated on the test split.


In [11]:
# ================================================================
# END-TO-END TASK EXPERIMENT
# Candidate set     = All attention heads
# Head ranking      = Fully adaptive guarded task-level joint refinement
# Steering signal  = W_O * E[a_fewshot - a_zeroshot]
# K / alpha tuning = Validation only
# Final result      = Test accuracy of the frozen validation-selected config
# ================================================================


# ================================================================
# Task-level joint refinement over arbitrary head subsets
# ================================================================

def _prepare_task_level_examples(
    prompt_specs: List[Dict[str, Any]],
    tokenizer: Any,
    device: torch.device,
) -> List[Dict[str, Any]]:
    examples = []

    for spec in prompt_specs:
        prompt = spec["prompt_zero"]
        target_str = spec["target_str"]
        target_text = " " + str(target_str).strip()

        prompt_ids = tokenizer.encode(
            prompt, add_special_tokens=False
        )
        completion_ids = tokenizer.encode(
            prompt + target_text,
            add_special_tokens=False,
        )

        if completion_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(
                f"Target changes tokenization across prompt boundary "
                f"for {target_str!r}."
            )

        target_ids = completion_ids[len(prompt_ids):]
        if not target_ids:
            raise ValueError(
                f"Empty target tokenization for {target_str!r}"
            )

        input_ids = torch.tensor(
            [prompt_ids + target_ids],
            device=device,
            dtype=torch.long,
        )
        attention_mask = torch.ones_like(input_ids)

        start = len(prompt_ids) - 1
        end = start + len(target_ids)

        examples.append(
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "start": int(start),
                "end": int(end),
                "target_ids": [int(x) for x in target_ids],
            }
        )

    return examples


class TaskLevelSubsetScorer:
    """
    Scores arbitrary subsets of the detected heads jointly on successful
    TRAIN zero-shot queries, using the same mean displacement vectors used
    by the final intervention.
    """

    def __init__(
        self,
        candidate_heads: List[Tuple[int, int]],
        mean_displacements: torch.Tensor,
        train_prompt_specs: List[Dict[str, Any]],
        model: Any,
        model_config: Dict[str, Any],
        tokenizer: Any,
        eval_batch_size: int = 8,
    ):
        self.candidate_heads = [tuple(h) for h in candidate_heads]
        self.head_to_idx = {
            h: i for i, h in enumerate(self.candidate_heads)
        }

        self.model = model
        self.model_config = model_config
        self.device = model_config["device"]
        self.dtype = next(model.parameters()).dtype
        self.eval_batch_size = max(1, int(eval_batch_size))

        self.examples = _prepare_task_level_examples(
            train_prompt_specs, tokenizer, self.device
        )
        self.cache: Dict[
            frozenset, Dict[str, float]
        ] = {}

        self.projected = project_head_displacement_vectors(
            core_heads=self.candidate_heads,
            head_displacements=mean_displacements,
            model=model,
            model_config=model_config,
            device=self.device,
            dtype=self.dtype,
        )

        self.layer_to_head_indices: Dict[
            int, List[int]
        ] = defaultdict(list)
        self.layer_to_vectors: Dict[
            int, torch.Tensor
        ] = {}

        for idx, (layer, head) in enumerate(
            self.candidate_heads
        ):
            self.layer_to_head_indices[
                layer
            ].append(idx)

        for layer, indices in (
            self.layer_to_head_indices.items()
        ):
            vectors = [
                self.projected[
                    self.candidate_heads[i]
                ].reshape(-1)
                for i in indices
            ]
            self.layer_to_vectors[
                layer
            ] = torch.stack(
                vectors, dim=0
            ).to(
                device=self.device,
                dtype=self.dtype,
            )

        self._mask = None
        self._start = None
        self._end = None

        self.handles = [
            get_module(
                model,
                model_config[
                    "layer_hook_names"
                ][layer],
            ).register_forward_hook(
                self._make_hook(layer)
            )
            for layer in sorted(
                self.layer_to_head_indices
            )
        ]

    def _make_hook(self, layer: int):
        indices = self.layer_to_head_indices[
            layer
        ]
        vectors = self.layer_to_vectors[
            layer
        ]

        def hook(module, args, output):
            if self._mask is None:
                return output

            is_tuple = isinstance(
                output, tuple
            )
            hidden = (
                output[0]
                if is_tuple
                else output
            )

            local_mask = self._mask[
                :, indices
            ].to(
                device=hidden.device,
                dtype=hidden.dtype,
            )
            local_vectors = vectors.to(
                device=hidden.device,
                dtype=hidden.dtype,
            )
            additions = (
                local_mask
                @ local_vectors
            )

            edited = hidden.clone()
            edited[
                :,
                self._start:self._end,
                :,
            ] = (
                edited[
                    :,
                    self._start:self._end,
                    :,
                ]
                + additions[:, None, :]
            )

            if is_tuple:
                return (
                    edited,
                ) + output[1:]
            return edited

        return hook

    def close(self):
        for handle in self.handles:
            handle.remove()

        self.handles = []

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _subset_mask(
        self,
        subsets: List[frozenset],
    ) -> torch.Tensor:
        mask = torch.zeros(
            len(subsets),
            len(self.candidate_heads),
            dtype=torch.float32,
            device=self.device,
        )

        for row, subset in enumerate(
            subsets
        ):
            for head in subset:
                mask[
                    row,
                    self.head_to_idx[
                        tuple(head)
                    ],
                ] = 1.0

        return mask

    def score_many(
        self,
        subsets: List[
            Set[Tuple[int, int]]
        ],
    ) -> Dict[
        frozenset, Dict[str, float]
    ]:
        keys = [
            frozenset(
                tuple(h)
                for h in subset
            )
            for subset in subsets
        ]

        missing = list(
            dict.fromkeys(
                k
                for k in keys
                if k not in self.cache
            )
        )

        for start_idx in range(
            0,
            len(missing),
            self.eval_batch_size,
        ):
            chunk = missing[
                start_idx:
                start_idx
                + self.eval_batch_size
            ]
            bsz = len(chunk)
            if bsz == 0:
                continue

            self._mask = (
                self._subset_mask(chunk)
            )

            sum_logp = torch.zeros(
                bsz,
                dtype=torch.float64,
            )
            correct_count = torch.zeros(
                bsz,
                dtype=torch.float64,
            )

            for ex in self.examples:
                self._start = ex["start"]
                self._end = ex["end"]

                ids = ex[
                    "input_ids"
                ].repeat(bsz, 1)
                attention_mask = ex[
                    "attention_mask"
                ].repeat(bsz, 1)

                target = torch.tensor(
                    ex["target_ids"],
                    device=self.device,
                    dtype=torch.long,
                )

                with torch.inference_mode():
                    logits = self.model(
                        input_ids=ids,
                        attention_mask=
                            attention_mask,
                    ).logits[
                        :,
                        self._start:
                        self._end,
                        :,
                    ]

                top1 = logits.argmax(
                    dim=-1
                )
                correct = (
                    top1
                    == target[None, :]
                ).all(dim=-1)

                log_probs = (
                    F.log_softmax(
                        logits.float(),
                        dim=-1,
                    )
                )

                positions = torch.arange(
                    len(
                        ex["target_ids"]
                    ),
                    device=self.device,
                )

                mean_target_logp = (
                    log_probs[
                        :,
                        positions,
                        target,
                    ].mean(dim=-1)
                )

                sum_logp += (
                    mean_target_logp
                    .detach()
                    .cpu()
                    .double()
                )
                correct_count += (
                    correct
                    .detach()
                    .cpu()
                    .double()
                )

            n_examples = max(
                1, len(self.examples)
            )

            for row, key in enumerate(
                chunk
            ):
                self.cache[key] = {
                    "correct_fraction":
                        float(
                            correct_count[
                                row
                            ].item()
                            / n_examples
                        ),
                    "mean_logp":
                        float(
                            sum_logp[
                                row
                            ].item()
                            / n_examples
                        ),
                }

        self._mask = None
        self._start = None
        self._end = None

        return {
            k: self.cache[k]
            for k in keys
        }


def build_task_level_greedy_binary_ranking(
    candidate_heads: List[
        Tuple[int, int]
    ],
    mean_displacements: torch.Tensor,
    train_prompt_specs: List[
        Dict[str, Any]
    ],
    model: Any,
    model_config: Dict[str, Any],
    tokenizer: Any,
    eval_batch_size: int = 32,
) -> Dict[str, Any]:
    """
    Greedy backward deletion over all model heads.

    At every step, rank all single-head removals by correctness fraction and
    mean target log-probability. Choose an adaptive coarse batch near one
    eighth of the current head count, rounded to the nearest power of two,
    then try that batch and successively halve it down to 2. A multi-head
    deletion is accepted only if it preserves the current exact
    teacher-forced correctness fraction. If no multi-head batch is safe,
    fall back to the best single-head removal.

    The same adaptive guarded rule is used throughout the ranking.
    """
    scorer = TaskLevelSubsetScorer(
        candidate_heads=
            candidate_heads,
        mean_displacements=
            mean_displacements,
        train_prompt_specs=
            train_prompt_specs,
        model=model,
        model_config=model_config,
        tokenizer=tokenizer,
        eval_batch_size=
            eval_batch_size,
    )

    try:
        current = set(
            tuple(h)
            for h in candidate_heads
        )

        if not current:
            raise ValueError(
                "No detected candidate heads."
            )

        removal_order: List[
            Tuple[int, int]
        ] = []

        current_stats = (
            scorer.score_many(
                [current]
            )[frozenset(current)]
        )

        pbar = tqdm(
            total=max(
                0, len(current) - 1
            ),
            desc=(
                "Task-level greedy "
                "pruning (all candidates)"
            ),
            mininterval=5.0,
        )

        while len(current) > 1:
            heads = sorted(current)
            children = [
                current - {head}
                for head in heads
            ]

            child_stats = (
                scorer.score_many(
                    children
                )
            )

            ranked_removals = []
            for head, child in zip(
                heads, children
            ):
                stats = child_stats[
                    frozenset(child)
                ]
                key = (
                    float(
                        stats[
                            "correct_fraction"
                        ]
                    ),
                    float(
                        stats[
                            "mean_logp"
                        ]
                    ),
                )
                ranked_removals.append(
                    (
                        key,
                        head,
                        child,
                        stats,
                    )
                )

            ranked_removals.sort(
                key=lambda x: x[0],
                reverse=True,
            )

            # ----------------------------------------------------
            # Adaptive guarded coarse-to-fine pruning at every k.
            #
            # Start near k / 8, rounded to the nearest power of 2:
            # 1024 -> 128, 512 -> 64, 60 -> 8, 32 -> 4, 16 -> 2.
            # Then fall back by halving until 2; if none are safe,
            # use the best single-head removal below.
            # ----------------------------------------------------
            k_current = len(current)
            target_batch = max(
                1.0,
                float(k_current) / 8.0,
            )
            start_exponent = int(
                math.floor(
                    math.log2(
                        target_batch
                    )
                    + 0.5
                )
            )
            start_batch = min(
                max(
                    1,
                    2 ** start_exponent,
                ),
                k_current - 1,
            )

            batch_sizes = []
            batch_size = int(
                start_batch
            )
            while batch_size >= 2:
                if (
                    batch_size
                    < k_current
                    and batch_size
                    <= len(ranked_removals)
                ):
                    batch_sizes.append(
                        batch_size
                    )
                batch_size //= 2

            accepted_batch = None
            accepted_child = None
            accepted_stats = None

            for batch_size in batch_sizes:
                batch_heads = [
                    item[1]
                    for item
                    in ranked_removals[
                        :batch_size
                    ]
                ]
                trial_child = (
                    current
                    - set(batch_heads)
                )

                trial_stats = (
                    scorer.score_many(
                        [trial_child]
                    )[
                        frozenset(
                            trial_child
                        )
                    ]
                )

                if (
                    trial_stats[
                        "correct_fraction"
                    ]
                    + 1e-12
                    >= current_stats[
                        "correct_fraction"
                    ]
                ):
                    accepted_batch = (
                        batch_heads
                    )
                    accepted_child = (
                        trial_child
                    )
                    accepted_stats = (
                        trial_stats
                    )
                    break

            if accepted_batch is not None:
                if (
                    len(accepted_batch)
                    < start_batch
                ):
                    tqdm.write(
                        f"[*] Adaptive pruning fallback: "
                        f"k={k_current}, proposed="
                        f"{start_batch}, accepted="
                        f"{len(accepted_batch)}"
                    )

                removal_order.extend(
                    tuple(h)
                    for h
                    in accepted_batch
                )
                current = set(
                    accepted_child
                )
                current_stats = dict(
                    accepted_stats
                )
                pbar.update(
                    len(accepted_batch)
                )
                continue

            if start_batch >= 2:
                tqdm.write(
                    f"[*] Adaptive pruning fallback: "
                    f"k={k_current}, proposed="
                    f"{start_batch}, accepted=1"
                )

            # ----------------------------------------------------
            # Single-head fallback when no safe multi-head deletion
            # is available (or when the adaptive rule returns 1).
            # ----------------------------------------------------
            (
                _,
                best_head,
                best_child,
                best_stats,
            ) = ranked_removals[0]

            removal_order.append(
                tuple(best_head)
            )
            current = set(
                best_child
            )
            current_stats = dict(
                best_stats
            )
            pbar.update(1)

        pbar.close()

        last_head = next(
            iter(current)
        )

        importance_order = [
            last_head
        ] + list(
            reversed(removal_order)
        )

        return {
            "search":
                (
                    "task_level_"
                    "fully_adaptive_guarded_"
                    "greedy_backward_"
                    "binary_all_heads"
                ),
            "candidate_count":
                int(
                    len(candidate_heads)
                ),
            "importance_order": [
                (int(l), int(h))
                for l, h
                in importance_order
            ],
            "removal_order": [
                (int(l), int(h))
                for l, h
                in removal_order
            ],
        }

    finally:
        scorer.close()


def adaptive_alpha_search(
    eval_fn,
    coarse_alphas: Optional[List[float]] = None,
    min_step: float = 0.125,
    stage2_segments: int = 12,
    stage3_segments: int = 4,
) -> Dict[str, Any]:
    """Bounded three-stage validation-only search over injection alpha."""
    if coarse_alphas is None:
        coarse_alphas = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

    coarse = sorted(
        {round(float(a), 8) for a in coarse_alphas}
    )
    if len(coarse) < 2 or coarse[0] < 0:
        raise ValueError(
            "coarse_alphas must contain at least two non-negative values."
        )

    results: Dict[float, float] = {}
    stages: Dict[float, str] = {}

    def evaluate(alphas, stage: str) -> None:
        pending = [
            a
            for a in sorted(
                {round(float(x), 8) for x in alphas}
            )
            if a not in results
        ]
        if not pending:
            return

        evaluated = eval_fn(pending)
        for a in pending:
            results[a] = float(evaluated[a])
            stages[a] = stage
            print(
                f"      • [Val:{stage}] Alpha: {a:06.3f} "
                f"-> Accuracy: {results[a] * 100.0:.2f}%"
            )

    evaluate(coarse, "coarse")

    coarse_best_acc = max(results[a] for a in coarse)
    coarse_best_alpha = min(
        a for a in coarse
        if results[a] == coarse_best_acc
    )
    best_idx = coarse.index(coarse_best_alpha)

    if best_idx == 0:
        left, right = coarse[0], coarse[1]
    elif best_idx == len(coarse) - 1:
        left, right = coarse[-2], coarse[-1]
    else:
        left, right = (
            coarse[best_idx - 1],
            coarse[best_idx + 1],
        )

    def bounded_grid(
        lo: float,
        hi: float,
        target_segments: int,
    ) -> List[float]:
        width = hi - lo
        if width <= 0:
            return [float(lo)]
        max_segments = int(math.floor(width / min_step))
        if max_segments < 2:
            return [float(lo), float(hi)]
        n_segments = min(target_segments, max_segments)
        return [
            round(float(a), 8)
            for a in np.linspace(
                lo, hi, n_segments + 1
            )
        ]

    before_stage2 = len(results)
    evaluate(
        bounded_grid(left, right, stage2_segments),
        "refine_1",
    )
    stage2_evals = len(results) - before_stage2

    local_alphas = sorted(
        a for a in results
        if left <= a <= right
    )
    local_best_acc = max(
        results[a] for a in local_alphas
    )
    local_best = min(
        a for a in local_alphas
        if results[a] == local_best_acc
    )
    local_idx = local_alphas.index(local_best)
    final_left = local_alphas[
        max(0, local_idx - 1)
    ]
    final_right = local_alphas[
        min(len(local_alphas) - 1, local_idx + 1)
    ]

    before_stage3 = len(results)
    if final_right > final_left:
        evaluate(
            bounded_grid(
                final_left,
                final_right,
                stage3_segments,
            ),
            "refine_2",
        )
    stage3_evals = len(results) - before_stage3

    best_acc = max(results.values())
    # Tie-break toward the smaller alpha.
    best_alpha = min(
        a for a, acc in results.items()
        if acc == best_acc
    )

    return {
        "best_alpha": float(best_alpha),
        "best_accuracy": float(best_acc),
        "coarse_best_alpha": float(coarse_best_alpha),
        "coarse_best_accuracy": float(coarse_best_acc),
        "refinement_bracket": [
            float(left), float(right)
        ],
        "final_refinement_bracket": [
            float(final_left), float(final_right)
        ],
        "stage2_evaluations": int(stage2_evals),
        "stage3_evaluations": int(stage3_evals),
        "results": dict(sorted(results.items())),
        "stages": dict(sorted(stages.items())),
    }


def run_task_experiment(
    dataset_name: str,
    model: Any,
    tokenizer: Any,
    model_config: Dict[str, Any],
    root_data_dir: str = "function_vectors/dataset_files",
    n_shots: int = 10,
    n_eval_prompts: int = 25,
    alpha_list: Optional[List[float]] = None,
    seed: int = 42,
    output_root: str = "outputs",
) -> Dict[str, Any]:
    """
    End-to-end task-specific displacement-based Function Vector experiment.

    TRAIN
      1. Collect successful TRAIN prompts for task-level refinement.
      2. Initialize task-level joint refinement with every attention head.
      3. Estimate E[a_fewshot-a_zeroshot] from successful TRAIN prompts only.
      4. Rank all model heads with fully adaptive guarded task-level
         backward deletion using the actual mean displacement vector.

    VALIDATION / TEST
      5. For each Top-k of the refined ranking, tune alpha on VALIDATION only.
      6. Select (k, alpha) by validation accuracy.
      7. Report TEST accuracy of that frozen configuration.
    """
    set_seed(seed)

    if alpha_list is None:
        alpha_list = [
            0.5, 1.0, 2.0, 4.0,
            8.0, 16.0, 32.0,
        ]

    print(
        f"\n{'=' * 70}\n"
        f"🚀 Running Experiment on Task: "
        f"{dataset_name.upper()}\n"
        f"{'=' * 70}"
    )

    dataset = load_dataset(
        dataset_name,
        root_data_dir=root_data_dir,
        seed=seed,
    )

    prefixes = {
        "input": "Q:",
        "output": "A:",
        "instructions": "",
    }
    separators = {
        "input": "\n",
        "output": "\n\n",
        "instructions": "",
    }
    pd_kwargs = {
        "prepend_bos_token": model_config["prepend_bos"],
        "bos_string": model_config["bos_string"],
        "prefixes": prefixes,
        "separators": separators,
    }

    model_tag = model_config["name_or_path"].split("/")[-1]
    model_slug = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        model_config["name_or_path"],
    )
    run_dir = os.path.join(
        output_root,
        "main_method",
        model_slug,
        f"seed_{seed}",
    )
    cache_dir = os.path.join(run_dir, "cache")
    plot_dir = os.path.join(run_dir, "plots")
    results_path = os.path.join(
        run_dir,
        f"main_method_results_{model_slug}_seed_{seed}.json",
    )
    os.makedirs(cache_dir, exist_ok=True)
    os.makedirs(plot_dir, exist_ok=True)

    n_train = len(dataset["train"])
    actual_shots = min(n_shots, n_train - 1)
    actual_eval_prompts = min(
        n_eval_prompts, n_train
    )

    if actual_shots < 1:
        raise ValueError(
            f"Training split too small for {n_shots}-shot ICL."
        )

    # ============================================================
    # 1. TRAIN: collect successful prompts for joint refinement
    # ============================================================
    prompt_cache_version = 1
    prompt_cache = os.path.join(
        cache_dir,
        (
            f"joint_prompt_specs_v{prompt_cache_version}_"
            f"{model_slug}_seed_{seed}_{dataset_name}.pt"
        ),
    )

    n_clean_base_fail = 0
    n_injection_base_fail = 0
    n_generation_fail = 0
    n_zero_head = 0
    successful_prompt_specs: List[
        Dict[str, Any]
    ] = []

    cache_is_current = False

    if os.path.exists(prompt_cache):
        cached = torch.load(
            prompt_cache, map_location="cpu"
        )
        cache_is_current = (
            cached.get("cache_version")
            == prompt_cache_version
            and cached.get("method")
            == "all_head_joint_refinement"
            and cached.get("split_protocol")
            == "60/20/20"
            and cached.get("seed")
            == int(seed)
            and cached.get("n_shots")
            == int(actual_shots)
            and cached.get("task")
            == dataset_name
            and cached.get("model")
            == model_config["name_or_path"]
            and "successful_prompt_specs"
            in cached
        )

        if cache_is_current:
            print(
                f"[*] Loading cached successful TRAIN "
                f"prompts from {prompt_cache}"
            )
            successful_prompt_specs = [
                {
                    "prompt_zero":
                        spec["prompt_zero"],
                    "target_str":
                        spec["target_str"],
                    "target_idx":
                        int(spec["target_idx"]),
                    "demonstration_indices": [
                        int(x)
                        for x in spec[
                            "demonstration_indices"
                        ]
                    ],
                }
                for spec
                in cached[
                    "successful_prompt_specs"
                ]
            ]
            n_clean_base_fail = int(
                cached.get(
                    "n_clean_base_fail", 0
                )
            )

    if not cache_is_current:
        rng = np.random.default_rng(seed)
        train_indices = rng.permutation(n_train)
        p_idx = 0
        accepted = 0

        while (
            accepted < actual_eval_prompts
            and p_idx < n_train
        ):
            target_idx = int(
                train_indices[p_idx]
            )
            target_pair = dataset["train"][
                target_idx
            ]
            target_str = (
                target_pair["output"][0]
                if isinstance(
                    target_pair["output"],
                    list,
                )
                else target_pair["output"]
            )

            remaining = np.array(
                [
                    i for i in range(n_train)
                    if i != target_idx
                ],
                dtype=int,
            )
            demo_indices = rng.choice(
                remaining,
                size=actual_shots,
                replace=False,
            )
            demos = dataset["train"][
                demo_indices
            ]

            prompt_icl = create_prompt(
                word_pairs_to_prompt_data(
                    demos,
                    query_target_pair=target_pair,
                    **pd_kwargs,
                )
            )
            prompt_neutral = create_prompt(
                word_pairs_to_prompt_data(
                    {"input": [], "output": []},
                    query_target_pair=target_pair,
                    **pd_kwargs,
                )
            )

            inputs = tokenizer(
                [prompt_icl],
                return_tensors="pt",
            ).to(model_config["device"])
            prompt_len = inputs.input_ids.shape[1]
            eval_max_new_tokens = max(
                3,
                len(
                    tokenizer.encode(
                        " " + str(target_str).strip(),
                        add_special_tokens=False,
                    )
                ),
            )

            with torch.inference_mode():
                generated_ids = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=
                        eval_max_new_tokens,
                    pad_token_id=
                        tokenizer.eos_token_id,
                )

            generated = clean_text(
                tokenizer.decode(
                    generated_ids[0][prompt_len:],
                    skip_special_tokens=True,
                )
            )

            if generated == clean_text(
                target_str
            ):
                successful_prompt_specs.append(
                    {
                        "prompt_zero":
                            prompt_neutral,
                        "target_str":
                            str(target_str),
                        "target_idx":
                            int(target_idx),
                        "demonstration_indices": [
                            int(x)
                            for x in demo_indices
                        ],
                    }
                )
                accepted += 1
            else:
                n_clean_base_fail += 1

            p_idx += 1

        if not successful_prompt_specs:
            print(
                "[-] No successful TRAIN prompts."
            )
            return {}

        torch.save(
            {
                "cache_version":
                    prompt_cache_version,
                "method":
                    "all_head_joint_refinement",
                "split_protocol":
                    "60/20/20",
                "task": dataset_name,
                "model":
                    model_config["name_or_path"],
                "seed": int(seed),
                "n_shots": int(actual_shots),
                "requested_eval_prompts":
                    int(actual_eval_prompts),
                "successful_prompt_specs":
                    successful_prompt_specs,
                "n_clean_base_fail":
                    int(n_clean_base_fail),
            },
            prompt_cache,
        )
        print(
            f"[*] Saved successful TRAIN prompt "
            f"cache to {prompt_cache}"
        )

    total_prompts = len(
        successful_prompt_specs
    )
    if total_prompts == 0:
        return {}

    # ============================================================
    # 2. Mean displacement
    # ============================================================
    displacement_cache = os.path.join(
        cache_dir,
        (
            f"mean_displacements_"
            f"{model_slug}_seed_{seed}_{dataset_name}.pt"
        ),
    )

    mean_displacements = (
        get_successful_mean_head_displacements(
            dataset=dataset,
            model=model,
            model_config=model_config,
            tokenizer=tokenizer,
            n_icl_examples=actual_shots,
            n_trials=100,
            prefixes=prefixes,
            separators=separators,
            seed=seed,
            cache_path=displacement_cache,
        )
    )

    # ============================================================
    # 3. Task-level joint refinement over all model heads
    # ============================================================
    candidate_heads = [
        (layer, head)
        for layer in range(
            model_config["n_layers"]
        )
        for head in range(
            model_config["n_heads"]
        )
    ]

    print(
        f"\n{'=' * 70}\n"
        f"🔬 TASK-LEVEL JOINT GREEDY REFINEMENT\n"
        f"{'=' * 70}\n"
        f"   Candidate source: all model heads\n"
        f"   Joint candidates: "
        f"{len(candidate_heads)}"
    )

    joint_search = (
        build_task_level_greedy_binary_ranking(
            candidate_heads=candidate_heads,
            mean_displacements=
                mean_displacements,
            train_prompt_specs=
                successful_prompt_specs,
            model=model,
            model_config=model_config,
            tokenizer=tokenizer,
            eval_batch_size=32,
        )
    )

    ranked_heads = [
        tuple(h)
        for h in joint_search[
            "importance_order"
        ]
    ]
    total_available_heads = len(
        ranked_heads
    )

    # ============================================================
    # 4. Validation / test evaluation sets
    # ============================================================
    valid_filter_set = (
        get_few_shot_filter_set(
            dataset,
            "valid",
            actual_shots,
            model,
            model_config,
            tokenizer,
            prefixes,
            separators,
            seed=seed,
        )
    )
    test_filter_set = (
        get_few_shot_filter_set(
            dataset,
            "test",
            actual_shots,
            model,
            model_config,
            tokenizer,
            prefixes,
            separators,
            seed=seed,
        )
    )

    if len(valid_filter_set) == 0:
        print(
            "[!] Warning: validation ICL filter is empty. "
            "Alpha/head-count selection is not informative."
        )
    if len(test_filter_set) == 0:
        print(
            "[!] Warning: test ICL filter is empty."
        )

    requested_head_counts = [
        1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        15, 20, 30, 40, 60,
    ]
    # Evaluate only fixed Top-K choices; the all-head set is never a candidate.
    head_counts_to_test = [
        k
        for k in requested_head_counts
        if k < total_available_heads
    ]
    head_counts_to_test = sorted(
        set(head_counts_to_test)
    )

    task_results = {
        "task": dataset_name,
        "model": model_config["name_or_path"],
        "seed": int(seed),
        "selection_signal":
            "all_model_heads",
        "vector_signal":
            "W_O(E[a_fewshot-a_zeroshot])",
        "head_selection_method":
            "all_model_heads_then_task_level_joint_greedy_binary",
        "head_score_unit":
            "task_level_binary_with_mean_logp_tiebreak",
        "task_level_candidate_count":
            int(len(candidate_heads)),
        "task_level_removal_order": [
            (int(l), int(h))
            for l, h in joint_search[
                "removal_order"
            ]
        ],
        "total_train_prompts":
            int(total_prompts),
        "requested_train_prompts":
            int(actual_eval_prompts),
        "n_clean_base_fail":
            int(n_clean_base_fail),
        "n_injection_base_fail":
            int(n_injection_base_fail),
        "n_generation_fail":
            int(n_generation_fail),
        "n_zero_head":
            int(n_zero_head),
        "icl_accuracy_valid_full":
            len(valid_filter_set)
            / max(1, len(dataset["valid"]))
            * 100.0,
        "icl_accuracy_test_full":
            len(test_filter_set)
            / max(1, len(dataset["test"]))
            * 100.0,
        "n_valid_eval":
            int(len(valid_filter_set)),
        "n_valid_total":
            int(len(dataset["valid"])),
        "n_test_eval":
            int(len(test_filter_set)),
        "n_test_total":
            int(len(dataset["test"])),
        "total_ranked_heads":
            int(total_available_heads),
        "head_counts_evaluated": [
            int(k)
            for k in head_counts_to_test
        ],
        "head_ranking": [
            {
                "rank": int(rank + 1),
                "layer": int(head[0]),
                "head": int(head[1]),
            }
            for rank, head
            in enumerate(ranked_heads)
        ],
        "validation_tuning": {},
        "head_count_curve": [],
        "final_test_evaluation": {},
    }

    # ============================================================
    # 5. VALIDATION: tune alpha independently for every Top-k
    # ============================================================
    print(
        f"\n{'=' * 70}\n"
        f"🔍 HYPERPARAMETER TUNING "
        f"(VALIDATION ONLY)\n"
        f"{'=' * 70}"
    )

    best_val_result = {
        "val_acc": -1.0,
        "num_heads": 0,
        "best_alpha": None,
        "heads": [],
    }

    for num_heads in head_counts_to_test:
        selected_heads = ranked_heads[
            :num_heads
        ]
        is_all_heads = (
            num_heads
            == total_available_heads
        )
        label = (
            "ALL"
            if is_all_heads
            else str(num_heads)
        )

        print(
            f"\n{'-' * 60}\n"
            f"📊 Validation | Top-{label} "
            f"Displacement Heads\n"
            f"{'-' * 60}"
        )

        layer_fvs_for_k = (
            compute_layerwise_function_vectors(
                core_heads=selected_heads,
                mean_displacements=
                    mean_displacements,
                model=model,
                model_config=model_config,
                device=model_config["device"],
                dtype=next(
                    model.parameters()
                ).dtype,
            )
        )

        def eval_alphas(
            alphas: List[float],
        ) -> Dict[float, float]:
            return (
                validate_head_fv_accuracies(
                    alphas=alphas,
                    core_heads=selected_heads,
                    mean_displacements=
                        mean_displacements,
                    dataset=dataset,
                    split_name="valid",
                    model=model,
                    model_config=model_config,
                    tokenizer=tokenizer,
                    prefixes=prefixes,
                    separators=separators,
                    filter_set=valid_filter_set,
                    layer_fvs=
                        layer_fvs_for_k,
                )
            )

        alpha_search = (
            adaptive_alpha_search(
                eval_alphas,
                coarse_alphas=alpha_list,
            )
        )

        tested_alphas = list(
            alpha_search["results"].keys()
        )
        val_accuracies = [
            alpha_search["results"][a]
            * 100.0
            for a in tested_alphas
        ]
        best_alpha_for_k = float(
            alpha_search["best_alpha"]
        )
        best_acc_for_k = float(
            alpha_search["best_accuracy"]
            * 100.0
        )

        # k is traversed ascending. Strict > means validation ties
        # prefer fewer heads.
        if (
            best_acc_for_k
            > best_val_result["val_acc"]
        ):
            best_val_result = {
                "val_acc":
                    best_acc_for_k,
                "num_heads":
                    int(len(selected_heads)),
                "best_alpha":
                    best_alpha_for_k,
                "heads":
                    list(selected_heads),
            }

        key = f"{num_heads}_heads"
        task_results[
            "validation_tuning"
        ][key] = {
            "active_heads_count":
                int(len(selected_heads)),
            "is_all_heads":
                bool(is_all_heads),
            "heads": [
                (int(l), int(h))
                for l, h in selected_heads
            ],
            "val_accuracies": {
                float(a): float(v)
                for a, v in zip(
                    tested_alphas,
                    val_accuracies,
                )
            },
            "best_val_acc":
                best_acc_for_k,
            "best_alpha":
                best_alpha_for_k,
            "alpha_search": {
                "coarse_grid": [
                    float(a)
                    for a in alpha_list
                ],
                "coarse_best_alpha":
                    float(
                        alpha_search[
                            "coarse_best_alpha"
                        ]
                    ),
                "coarse_best_accuracy":
                    float(
                        alpha_search[
                            "coarse_best_accuracy"
                        ]
                        * 100.0
                    ),
                "refinement_bracket":
                    alpha_search[
                        "refinement_bracket"
                    ],
                "final_refinement_bracket":
                    alpha_search[
                        "final_refinement_bracket"
                    ],
                "stage2_evaluations":
                    int(
                        alpha_search[
                            "stage2_evaluations"
                        ]
                    ),
                "stage3_evaluations":
                    int(
                        alpha_search[
                            "stage3_evaluations"
                        ]
                    ),
                "stage_by_alpha": {
                    float(a):
                        alpha_search[
                            "stages"
                        ][a]
                    for a
                    in tested_alphas
                },
            },
        }

        plt.figure(
            figsize=(9, 5.5),
            dpi=120,
        )
        plt.plot(
            tested_alphas,
            val_accuracies,
            marker="o",
            linewidth=2.0,
        )
        plt.title(
            (
                f"Task: "
                f"{dataset_name.upper()} "
                f"[Validation] | "
                f"Top-{label} "
                f"Displacement Heads"
            )
        )
        plt.xlabel(
            "Steering Multiplier (Alpha)"
        )
        plt.ylabel(
            "Validation Accuracy (%)"
        )
        plt.xticks(
            tested_alphas,
            rotation=45,
        )
        plt.ylim(-5, 105)
        plt.grid(
            True,
            linestyle="--",
            alpha=0.5,
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(
                plot_dir,
                (
                    f"validation_alpha_"
                    f"{model_slug}_seed_{seed}_"
                    f"{dataset_name}_top_{num_heads}.png"
                ),
            ),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()

        del layer_fvs_for_k
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if (
        best_val_result["best_alpha"]
        is None
    ):
        raise RuntimeError(
            "No validation configuration "
            "was evaluated."
        )

    print(
        f"\n{'=' * 70}\n"
        f"✅ VALIDATION-SELECTED CONFIGURATION\n"
        f"{'=' * 70}\n"
        f"   Heads        : "
        f"{best_val_result['num_heads']}\n"
        f"   Alpha        : "
        f"{best_val_result['best_alpha']:.3f}\n"
        f"   Val Accuracy : "
        f"{best_val_result['val_acc']:.2f}%\n"
        f"{'=' * 70}"
    )

    # ============================================================
    # 6. TEST: frozen validation-tuned configs
    # ============================================================
    head_curve = []

    print(
        f"\n{'=' * 70}\n"
        f"🎯 TEST EVALUATION PER HEAD-COUNT LEVEL\n"
        f"{'=' * 70}"
    )

    for key, info in task_results[
        "validation_tuning"
    ].items():
        heads = [
            tuple(h)
            for h in info["heads"]
        ]
        alpha = float(
            info["best_alpha"]
        )
        n_heads_lvl = int(
            info["active_heads_count"]
        )
        is_all = bool(
            info["is_all_heads"]
        )
        label = (
            "ALL"
            if is_all
            else str(n_heads_lvl)
        )

        layer_fvs = (
            compute_layerwise_function_vectors(
                core_heads=heads,
                mean_displacements=
                    mean_displacements,
                model=model,
                model_config=model_config,
                device=model_config["device"],
                dtype=next(
                    model.parameters()
                ).dtype,
            )
        )

        test_acc = (
            validate_head_fv_accuracy(
                core_heads=heads,
                mean_displacements=
                    mean_displacements,
                dataset=dataset,
                split_name="test",
                model=model,
                model_config=model_config,
                tokenizer=tokenizer,
                prefixes=prefixes,
                separators=separators,
                filter_set=test_filter_set,
                alpha=alpha,
                layer_fvs=layer_fvs,
            )
            * 100.0
        )

        head_curve.append(
            {
                "num_heads":
                    n_heads_lvl,
                "is_all_heads":
                    is_all,
                "best_alpha":
                    alpha,
                "val_accuracy":
                    float(
                        info[
                            "best_val_acc"
                        ]
                    ),
                "test_accuracy":
                    float(test_acc),
                "heads": [
                    (int(l), int(h))
                    for l, h in heads
                ],
            }
        )

        print(
            f"      • Heads: {label:>3} "
            f"| α={alpha:>7.3f} "
            f"| Val: "
            f"{info['best_val_acc']:6.2f}% "
            f"| Test: {test_acc:6.2f}%"
        )

        del layer_fvs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    head_curve.sort(
        key=lambda x: x["num_heads"]
    )
    task_results[
        "head_count_curve"
    ] = head_curve

    # ============================================================
    # 7. Accuracy-vs-head-count plot
    # ============================================================
    xs = [
        c["num_heads"]
        for c in head_curve
    ]
    ys_test = [
        c["test_accuracy"]
        for c in head_curve
    ]
    ys_val = [
        c["val_accuracy"]
        for c in head_curve
    ]

    plt.figure(
        figsize=(10, 6),
        dpi=120,
    )
    plt.plot(
        xs,
        ys_test,
        marker="o",
        linewidth=2.5,
        label="Test accuracy",
    )
    plt.plot(
        xs,
        ys_val,
        marker="s",
        linewidth=1.8,
        linestyle="--",
        label="Validation accuracy",
    )
    plt.axhline(
        task_results[
            "icl_accuracy_test_full"
        ],
        linestyle=":",
        linewidth=1.8,
        label=(
            f"ICL {actual_shots}-shot "
            f"(full test): "
            f"{task_results['icl_accuracy_test_full']:.1f}%"
        ),
    )

    for item in head_curve:
        annotation = (
            f"α={item['best_alpha']:.2f}"
        )
        if item["is_all_heads"]:
            annotation += "\nALL"
        plt.annotate(
            annotation,
            (
                item["num_heads"],
                item["test_accuracy"],
            ),
            textcoords="offset points",
            xytext=(0, 9),
            fontsize=7,
            ha="center",
        )

    plt.title(
        (
            f"Task: {dataset_name.upper()} | "
            f"Displacement FV Accuracy vs Heads"
        )
    )
    plt.xlabel(
        "Number of Top-Ranked Heads"
    )
    plt.ylabel("Accuracy (%)")
    if len(set(xs)) > 1:
        plt.xscale("log")
    plt.xticks(
        xs,
        [
            "ALL"
            if c["is_all_heads"]
            else str(c["num_heads"])
            for c in head_curve
        ],
    )
    plt.ylim(-5, 105)
    plt.grid(
        True,
        linestyle="--",
        alpha=0.5,
        which="both",
    )
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            plot_dir,
            (
                f"accuracy_vs_head_count_"
                f"{model_slug}_seed_{seed}_"
                f"{dataset_name}.png"
            ),
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

    # ============================================================
    # 8. Final validation-selected result
    # ============================================================
    selected_num_heads = int(
        best_val_result["num_heads"]
    )
    selected_alpha = float(
        best_val_result["best_alpha"]
    )
    selected_heads = list(
        best_val_result["heads"]
    )

    selected_entry = next(
        (
            c for c in head_curve
            if c["num_heads"]
            == selected_num_heads
        ),
        None,
    )
    if selected_entry is None:
        raise RuntimeError(
            "Validation-selected head count "
            "missing from test curve."
        )

    final_test_acc_pct = float(
        selected_entry["test_accuracy"]
    )

    oracle = (
        max(
            head_curve,
            key=lambda x:
                x["test_accuracy"],
        )
        if head_curve
        else None
    )

    # ============================================================
    # 9. Save final displacement-based FV
    # ============================================================
    best_layer_fvs = (
        compute_layerwise_function_vectors(
            core_heads=selected_heads,
            mean_displacements=
                mean_displacements,
            model=model,
            model_config=model_config,
            device=model_config["device"],
            dtype=next(
                model.parameters()
            ).dtype,
        )
    )

    layer_fvs_cpu = {
        int(L):
            v.squeeze(0).squeeze(0)
            .float().cpu()
        for L, v
        in best_layer_fvs.items()
    }

    # This aggregate is analysis metadata only. Actual inference
    # remains layer-wise and does not inject this sum at one layer.
    fv_sum = (
        torch.stack(
            list(
                layer_fvs_cpu.values()
            )
        ).sum(dim=0)
        if layer_fvs_cpu
        else torch.zeros(
            model_config["resid_dim"]
        )
    )

    fv_path = os.path.join(
        cache_dir,
        (
            f"function_vector_"
            f"{model_slug}_seed_{seed}_"
            f"{dataset_name}.pt"
        ),
    )

    torch.save(
        {
            "task": dataset_name,
            "model":
                model_config["name_or_path"],
            "seed": int(seed),
            "selection_signal":
                "all_model_heads",
            "vector_signal":
                "W_O(E[a_fewshot-a_zeroshot])",
            "projection_bias_used":
                False,
            "head_selection_method":
                "all_model_heads_then_task_level_joint_greedy_binary",
            "head_score_unit":
                "task_level_binary_with_mean_logp_tiebreak",
            "task_level_candidate_count":
                int(len(candidate_heads)),
            "task_level_removal_order": [
                (int(l), int(h))
                for l, h in joint_search[
                    "removal_order"
                ]
            ],
            "num_heads":
                selected_num_heads,
            "core_heads": [
                (int(l), int(h))
                for l, h
                in selected_heads
            ],
            "ranked_heads": [
                (int(l), int(h))
                for l, h
                in ranked_heads
            ],
            "alpha": selected_alpha,
            "validation_accuracy":
                float(
                    best_val_result[
                        "val_acc"
                    ]
                ),
            "test_accuracy":
                final_test_acc_pct,
            "layer_fvs":
                layer_fvs_cpu,
            "fv_sum":
                fv_sum,
            "mean_displacements":
                mean_displacements.cpu(),
            "head_count_curve":
                head_curve,
        },
        fv_path,
    )
    print(
        f"[*] Saved displacement-based "
        f"function vector to {fv_path}"
    )

    # ============================================================
    # 10. Final results
    # ============================================================
    task_results[
        "final_test_evaluation"
    ] = {
        "test_accuracy":
            final_test_acc_pct,
        "validation_accuracy":
            float(
                best_val_result[
                    "val_acc"
                ]
            ),
        "selected_alpha":
            selected_alpha,
        "selected_num_heads":
            selected_num_heads,
        "selected_heads": [
            (int(l), int(h))
            for l, h
            in selected_heads
        ],
        "selection_signal":
            "all_model_heads",
        "vector_signal":
            "W_O(E[a_fewshot-a_zeroshot])",
        "oracle_test_accuracy": (
            float(
                oracle[
                    "test_accuracy"
                ]
            )
            if oracle
            else None
        ),
        "oracle_num_heads": (
            int(
                oracle[
                    "num_heads"
                ]
            )
            if oracle
            else None
        ),
        "oracle_alpha": (
            float(
                oracle[
                    "best_alpha"
                ]
            )
            if oracle
            else None
        ),
    }

    print(
        f"\n{'=' * 72}\n"
        f"🏆 FINAL RESULT FOR TASK: "
        f"{dataset_name.upper()}\n"
        f"{'=' * 72}\n"
        f"   ➤ FV Test Accuracy       : "
        f"{final_test_acc_pct:.2f}% "
        f"(filtered, n={len(test_filter_set)})\n"
        f"   ➤ Validation Accuracy    : "
        f"{best_val_result['val_acc']:.2f}% "
        f"(filtered, n={len(valid_filter_set)})\n"
        f"   ➤ ICL {actual_shots}-shot "
        f"(full test) : "
        f"{task_results['icl_accuracy_test_full']:.2f}%\n"
        f"   ➤ Selected # Heads       : "
        f"{selected_num_heads}\n"
        f"   ➤ Total Ranked Heads     : "
        f"{total_available_heads}\n"
        f"   ➤ Optimal Alpha (VAL)    : "
        f"{selected_alpha:.3f}\n"
        f"   ➤ Candidate Source       : "
        f"All model heads\n"
        f"   ➤ Joint Candidates       : "
        f"{len(candidate_heads)}\n"
        f"   ➤ Global Refinement      : "
        f"Fully adaptive guarded greedy deletion\n"
        f"   ➤ Joint-Search Prompts   : "
        f"{total_prompts} successful | "
        f"{n_clean_base_fail} clean fail\n"
        f"   ➤ Selected Core Heads    : "
        f"{selected_heads}\n"
        f"{'=' * 72}\n"
    )

    if oracle is not None:
        print(
            f"   [Analysis-only TEST oracle] "
            f"{oracle['test_accuracy']:.2f}% "
            f"@ {oracle['num_heads']} heads, "
            f"α={oracle['best_alpha']:.3f}"
        )

    save_experiment_results(
        task_results,
        filepath=results_path,
    )

    del (
        mean_displacements,
        successful_prompt_specs,
        dataset,
        best_layer_fvs,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return task_results


# 9. Experiment Runner

Outputs are stored under model- and seed-specific `outputs/main_method/` directories so runs with different models or seeds remain isolated.


In [12]:
import traceback


def run_all(
    model_names: List[str],
    tasks: Optional[List[str]] = None,
    seed: int = 42,
    n_shots: int = 10,
    n_eval_prompts: int = 25,
    alpha_list: Optional[List[float]] = None,
    root_data_dir: str = "function_vectors/dataset_files",
    output_root: str = "outputs",
):
    if tasks is None:
        tasks = [
            "antonym",
            "capitalize",
            "present-past",
            "singular-plural",
            "country-capital",
            "english-french"
        ]

    _env_tasks = os.environ.get("FV_TASKS", "").strip()
    if _env_tasks:
        tasks = [t.strip() for t in _env_tasks.split(",") if t.strip()]
        print(f"[config] FV_TASKS override -> {tasks}")

    if alpha_list is None:
        # Passed to run_task_experiment as the adaptive search's coarse grid.
        alpha_list = [0.5,1,2,4,8,16,32]

    grid = {}

    for model_name in model_names:
        tag = model_name.split("/")[-1]
        model_slug = re.sub(
            r"[^A-Za-z0-9._-]+",
            "_",
            model_name,
        )
        run_dir = os.path.join(
            output_root,
            "main_method",
            model_slug,
            f"seed_{seed}",
        )
        log_dir = os.path.join(run_dir, "logs")
        os.makedirs(log_dir, exist_ok=True)
        log_path = os.path.join(
            log_dir,
            f"main_method_{model_slug}_seed_{seed}.log",
        )

        print(f"\n{'#' * 70}")
        print(f"# MODEL: {model_name}")
        print(f"{'#' * 70}\n")

        model = tokenizer = model_config = None
        try:
            setup_logging(log_path)
            set_seed(seed)
            model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
        except KeyboardInterrupt:
            print("\n[!] Interrupted by user.")
            break
        except Exception as e:
            print(f"[!] Could not load '{model_name}': {e!r}")
            traceback.print_exc()
            grid[tag] = {t: None for t in tasks}
            continue

        model_results, failed = {}, []

        for task in tasks:
            try:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
                set_seed(seed)

                model_results[task] = run_task_experiment(
                    dataset_name=task,
                    model=model,
                    tokenizer=tokenizer,
                    model_config=model_config,
                    root_data_dir=root_data_dir,
                    n_shots=n_shots,
                    n_eval_prompts=n_eval_prompts,
                    alpha_list=alpha_list,
                    seed=seed,
                    output_root=output_root,
                )
            except KeyboardInterrupt:
                print("\n[!] Interrupted by user.")
                failed.append((task, "KeyboardInterrupt"))
                break
            except Exception as e:
                failed.append((task, repr(e)))
                print(f"[!] Error running task '{task}': {e!r}")
                traceback.print_exc()
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

        print(f"\n{'=' * 70}")
        print(f"RUN SUMMARY — {model_name}")
        print(f"{'=' * 70}")
        for task, res in model_results.items():
            acc = res.get("final_test_evaluation", {}).get("test_accuracy")
            print(f"   {task:<22} {acc:.2f}%" if acc is not None else f"   {task:<22} (no result)")
        for task, err in failed:
            print(f"   {task:<22} FAILED: {err}")
        print(f"{'=' * 70}\n")

        grid[tag] = {
            t: model_results.get(t, {}).get("final_test_evaluation", {}).get("test_accuracy")
            for t in tasks
        }

        del model, tokenizer, model_config
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    print(f"\n{'#' * 70}")
    print("# FINAL GRID — test accuracy (%)")
    print(f"{'#' * 70}")
    header = f"{'task':<22}" + "".join(f"{m:>20}" for m in grid.keys())
    print(header)
    print("-" * len(header))
    for t in tasks:
        row = f"{t:<22}"
        for m in grid.keys():
            v = grid[m].get(t)
            row += f"{v:>19.2f} " if v is not None else f"{'—':>20}"
        print(row)
    print(f"{'#' * 70}\n")

    model_group_slug = "__".join(
        re.sub(
            r"[^A-Za-z0-9._-]+",
            "_",
            model_name,
        )
        for model_name in model_names
    )
    summary_dir = os.path.join(
        output_root,
        "main_method",
        "summaries",
    )
    os.makedirs(summary_dir, exist_ok=True)
    grid_path = os.path.join(
        summary_dir,
        (
            f"main_method_test_accuracy_grid_"
            f"{model_group_slug}_seed_{seed}.json"
        ),
    )

    with open(grid_path, "w", encoding="utf-8") as f:
        json.dump(grid, f, indent=4)
    print(f"[*] Final test-accuracy grid saved to: {grid_path}")

    return grid


if __name__ == "__main__":
    run_all([RUN_MODEL], seed=RUN_SEED)



######################################################################
# MODEL: mistralai/Mistral-7B-v0.3
######################################################################

[*] Logging initialized. Output will be saved to: outputs/main_method/mistralai_Mistral-7B-v0.3/seed_43/logs/main_method_mistralai_Mistral-7B-v0.3_seed_43.log
[*] Loading Model: mistralai/Mistral-7B-v0.3


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

[*] Calculating 10-Shot Baseline Accuracy on 'VALID' split...


Finding Accurate ICL Prompts (valid): 100%|██████████| 940/940 [03:40<00:00,  4.27it/s]


[*] Found 745 / 940 successful baseline prompts on 'valid'.
[*] Calculating 10-Shot Baseline Accuracy on 'TEST' split...


Finding Accurate ICL Prompts (test): 100%|██████████| 940/940 [03:40<00:00,  4.26it/s]


[*] Found 720 / 940 successful baseline prompts on 'test'.

🔍 HYPERPARAMETER TUNING (VALIDATION ONLY)

------------------------------------------------------------
📊 Validation | Top-1 Displacement Heads
------------------------------------------------------------
      • [Val:coarse] Alpha: 00.500 -> Accuracy: 0.00%
      • [Val:coarse] Alpha: 01.000 -> Accuracy: 0.00%
      • [Val:coarse] Alpha: 02.000 -> Accuracy: 0.00%
      • [Val:coarse] Alpha: 04.000 -> Accuracy: 1.88%
      • [Val:coarse] Alpha: 08.000 -> Accuracy: 12.21%
      • [Val:coarse] Alpha: 16.000 -> Accuracy: 16.64%
      • [Val:coarse] Alpha: 32.000 -> Accuracy: 2.28%
      • [Val:refine_1] Alpha: 10.000 -> Accuracy: 13.83%
      • [Val:refine_1] Alpha: 12.000 -> Accuracy: 16.78%
      • [Val:refine_1] Alpha: 14.000 -> Accuracy: 17.05%
      • [Val:refine_1] Alpha: 18.000 -> Accuracy: 14.50%
      • [Val:refine_1] Alpha: 20.000 -> Accuracy: 12.48%
      • [Val:refine_1] Alpha: 22.000 -> Accuracy: 9.40%
      • [Val:r